In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not enabled!")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [6]:
!pip uninstall -y diffusers transformers tokenizers huggingface-hub -q

print("Old conflicting Hugging Face packages removed.")

Old conflicting Hugging Face packages removed.


In [7]:
!pip install -q \
    "transformers==4.57.6" \
    datasets \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    torchaudio \
    kagglehub \
    pandas \
    scikit-learn

print("ASR packages installation completed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
ASR packages installation completed.


In [8]:
# ============================================================
# CELL 3 - VERIFY ASR ENVIRONMENT
# ============================================================

import torch
import transformers
import datasets
import evaluate
import jiwer

print("========== ELDERCARE ASR ENVIRONMENT ==========")

print("PyTorch version       :", torch.__version__)
print("Transformers version  :", transformers.__version__)
print("Datasets version      :", datasets.__version__)
print("Evaluate version      :", evaluate.__version__)
print("Jiwer                 : Installed")

print("\n========== GPU CHECK ==========")

if torch.cuda.is_available():
    print("CUDA available        : True")
    print("GPU                   :", torch.cuda.get_device_name(0))
    print("CUDA version          :", torch.version.cuda)
else:
    print("CUDA available        : False")
    print("GPU                   : Not available")

print("\n========== IMPORT CHECK ==========")
print("PyTorch               : OK")
print("Transformers          : OK")
print("Datasets              : OK")
print("Evaluate              : OK")
print("Jiwer                 : OK")

print("\nASR environment verification completed successfully.")

========== ELDERCARE ASR ENVIRONMENT ==========
PyTorch version       : 2.11.0+cu128
Transformers version  : 5.16.1
Datasets version      : 4.0.0
Evaluate version      : 0.4.6
Jiwer                 : Installed

========== GPU CHECK ==========
CUDA available        : True
GPU                   : Tesla T4
CUDA version          : 12.8

========== IMPORT CHECK ==========
PyTorch               : OK
Transformers          : OK
Datasets              : OK
Evaluate              : OK
Jiwer                 : OK

ASR environment verification completed successfully.


In [9]:
# ============================================================
# CELL 4 - LOAD MMS MULTILINGUAL ASR MODEL
# ============================================================

import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

MODEL_NAME = "facebook/mms-1b-all"

print("========== LOADING MMS ASR MODEL ==========")
print("Model:", MODEL_NAME)
print("Please wait... The model is approximately 4 GB.")

# Load processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Load model
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True
)

# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move model to GPU
model = model.to(device)

# Evaluation mode
model.eval()

print("\n========== MODEL STATUS ==========")
print("Model loaded successfully.")
print("Device :", device)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

print("\n✓ MMS multilingual ASR model is ready.")

========== LOADING MMS ASR MODEL ==========
Model: facebook/mms-1b-all
Please wait... The model is approximately 4 GB.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]


========== MODEL STATUS ==========
Model loaded successfully.
Device : cuda
GPU    : Tesla T4

✓ MMS multilingual ASR model is ready.


In [10]:
# ============================================================
# CELL 5 - CONFIGURE MULTILINGUAL LANGUAGE ADAPTERS
# ============================================================

LANGUAGES = {
    "English": "eng",
    "Tamil": "tam",
    "Malayalam": "mal"
}

print("========== SUPPORTED LANGUAGES ==========")

for language, code in LANGUAGES.items():
    print(f"{language:10} -> {code}")

# Start with English adapter
processor.tokenizer.set_target_lang("eng")
model.load_adapter("eng")

print("\n========== ACTIVE LANGUAGE ==========")
print("Language : English")
print("Code     : eng")

print("\n✓ English MMS language adapter loaded successfully.")

========== SUPPORTED LANGUAGES ==========
English    -> eng
Tamil      -> tam
Malayalam  -> mal

========== ACTIVE LANGUAGE ==========
Language : English
Code     : eng

✓ English MMS language adapter loaded successfully.


In [11]:
# ============================================================
# CELL 6 - VERIFY MMS AUDIO CONFIGURATION
# ============================================================

import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

sampling_rate = processor.feature_extractor.sampling_rate

print("========== MMS AUDIO CONFIGURATION ==========")

print("Required sampling rate :", sampling_rate)
print("Current device          :", device)

if sampling_rate == 16000:
    print("\n✓ MMS expects 16 kHz audio.")
else:
    print("\n⚠ Unexpected sampling rate detected.")

print("\n✓ Audio configuration verification completed.")

========== MMS AUDIO CONFIGURATION ==========
Required sampling rate : 16000
Current device          : cuda

✓ MMS expects 16 kHz audio.

✓ Audio configuration verification completed.


In [12]:
# ============================================================
# CELL 7 - CHECK STORAGE AND GPU MEMORY
# ============================================================

import os
import shutil
import torch

print("========== STORAGE CHECK ==========")

total, used, free = shutil.disk_usage("/content")

GB = 1024 ** 3

print(f"Total storage : {total / GB:.2f} GB")
print(f"Used storage  : {used / GB:.2f} GB")
print(f"Free storage  : {free / GB:.2f} GB")

print("\n========== GPU MEMORY CHECK ==========")

if torch.cuda.is_available():

    gpu = torch.cuda.get_device_properties(0)

    print("GPU           :", gpu.name)
    print(f"GPU memory    : {gpu.total_memory / GB:.2f} GB")

else:
    print("GPU not available.")

print("\n✓ System resource check completed.")

========== STORAGE CHECK ==========
Total storage : 112.64 GB
Used storage  : 51.08 GB
Free storage  : 61.54 GB

========== GPU MEMORY CHECK ==========
GPU           : Tesla T4
GPU memory    : 14.56 GB

✓ System resource check completed.


In [13]:
# ============================================================
# CELL 8 - KAGGLE DATASET DOWNLOAD SETUP
# ============================================================

import os
import kagglehub

BASE_DIR = "/content/eldercare_kaggle_data"

os.makedirs(BASE_DIR, exist_ok=True)

print("========== KAGGLE DATASET SETUP ==========")
print("Download directory :", BASE_DIR)
print("KaggleHub          : OK")

print("\n✓ Ready to download both Kaggle datasets.")

========== KAGGLE DATASET SETUP ==========
Download directory : /content/eldercare_kaggle_data
KaggleHub          : OK

✓ Ready to download both Kaggle datasets.


In [14]:
# ============================================================
# CELL 9 - DOWNLOAD ENGLISH LIBRISPEECH KAGGLE DATASET
# ============================================================

import os
import kagglehub

ENGLISH_DATASET = "victorling/librispeech-clean"

print("========== DOWNLOADING ENGLISH DATASET ==========")
print("Dataset:", ENGLISH_DATASET)
print("Please wait...")

english_kaggle_path = kagglehub.dataset_download(
    ENGLISH_DATASET
)

print("\n========== DOWNLOAD COMPLETED ==========")
print("Dataset path:")
print(english_kaggle_path)

print("\n✓ English LibriSpeech Kaggle dataset downloaded successfully.")

========== DOWNLOADING ENGLISH DATASET ==========
Dataset: victorling/librispeech-clean
Please wait...


100%|██████████| 28.1G/28.1G [07:18<00:00, 68.8MB/s]

Extracting files...



========== DOWNLOAD COMPLETED ==========
Dataset path:
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1

✓ English LibriSpeech Kaggle dataset downloaded successfully.


In [15]:
# ============================================================
# CELL 10 - INSPECT ENGLISH KAGGLE DATASET
# ============================================================

import os

ENGLISH_KAGGLE_PATH = (
    "/root/.cache/kagglehub/datasets/"
    "victorling/librispeech-clean/versions/1"
)

print("========== ENGLISH KAGGLE DATASET INSPECTION ==========")
print("Dataset path:")
print(ENGLISH_KAGGLE_PATH)

print("\n========== TOP-LEVEL CONTENT ==========")

items = os.listdir(ENGLISH_KAGGLE_PATH)

for item in items[:30]:
    full_path = os.path.join(
        ENGLISH_KAGGLE_PATH,
        item
    )

    if os.path.isdir(full_path):
        print("[DIR] ", item)
    else:
        size_mb = os.path.getsize(full_path) / (1024 ** 2)
        print(f"[FILE] {item} ({size_mb:.2f} MB)")

print("\nTotal top-level items :", len(items))

print("\n✓ Dataset inspection completed.")

========== ENGLISH KAGGLE DATASET INSPECTION ==========
Dataset path:
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1

========== TOP-LEVEL CONTENT ==========
[DIR]  LibriSpeech

Total top-level items : 1

✓ Dataset inspection completed.


In [16]:
# ============================================================
# CELL 11 - INSPECT LIBRISPEECH KAGGLE STRUCTURE
# ============================================================

import os

LIBRISPEECH_DIR = os.path.join(
    ENGLISH_KAGGLE_PATH,
    "LibriSpeech"
)

print("========== LIBRISPEECH STRUCTURE ==========")
print("Path:", LIBRISPEECH_DIR)

if not os.path.exists(LIBRISPEECH_DIR):
    raise FileNotFoundError(
        "LibriSpeech directory was not found."
    )

# Find speaker directories
speaker_dirs = sorted([
    item for item in os.listdir(LIBRISPEECH_DIR)
    if os.path.isdir(
        os.path.join(LIBRISPEECH_DIR, item)
    )
])

print("Speaker folders found :", len(speaker_dirs))
print("First 10 speakers     :", speaker_dirs[:10])

# Find chapter directories
chapter_dirs = []

for speaker in speaker_dirs[:10]:

    speaker_path = os.path.join(
        LIBRISPEECH_DIR,
        speaker
    )

    chapters = sorted([
        item for item in os.listdir(speaker_path)
        if os.path.isdir(
            os.path.join(speaker_path, item)
        )
    ])

    for chapter in chapters:
        chapter_dirs.append(
            os.path.join(speaker_path, chapter)
        )

print("\nChapter folders inspected :", len(chapter_dirs))

if chapter_dirs:

    sample_chapter = chapter_dirs[0]

    print("\n========== SAMPLE CHAPTER ==========")
    print("Chapter:", sample_chapter)

    files = sorted(os.listdir(sample_chapter))

    print("Files in chapter:")

    for file in files[:20]:
        print("  ", file)

    flac_files = [
        f for f in files
        if f.endswith(".flac")
    ]

    transcript_files = [
        f for f in files
        if f.endswith(".trans.txt")
    ]

    print("\n========== FILE CHECK ==========")
    print("Audio files      :", len(flac_files))
    print("Transcript files :", len(transcript_files))

    if transcript_files:

        transcript_path = os.path.join(
            sample_chapter,
            transcript_files[0]
        )

        print("\n========== SAMPLE TRANSCRIPTS ==========")

        with open(
            transcript_path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f.readlines()[:5]:
                print(line.strip())

        print("\n✓ Audio + transcript structure detected.")

    else:

        print("\n⚠ No transcript file found in sample chapter.")

else:

    print("\n⚠ No chapter folders found.")

print("\n✓ Dataset structure inspection completed.")

========== LIBRISPEECH STRUCTURE ==========
Path: /root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech
Speaker folders found : 4
First 10 speakers     : ['dev-clean', 'test-clean', 'train-clean-100', 'train-clean-360']

Chapter folders inspected : 1252

========== SAMPLE CHAPTER ==========
Chapter: /root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/dev-clean/1272
Files in chapter:
   128104
   135031
   141231

========== FILE CHECK ==========
Audio files      : 0
Transcript files : 0

⚠ No transcript file found in sample chapter.

✓ Dataset structure inspection completed.


In [17]:
# ============================================================
# CELL 12 - INSPECT TRAIN-CLEAN-100 CORRECTLY
# ============================================================

import os

TRAIN_DIR = os.path.join(
    LIBRISPEECH_DIR,
    "train-clean-100"
)

print("========== TRAIN-CLEAN-100 INSPECTION ==========")
print("Path:", TRAIN_DIR)

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(
        "train-clean-100 directory was not found."
    )

# Get speaker folders
speakers = sorted([
    item for item in os.listdir(TRAIN_DIR)
    if os.path.isdir(
        os.path.join(TRAIN_DIR, item)
    )
])

print("Speaker folders found :", len(speakers))
print("First 10 speakers     :", speakers[:10])

# Select first speaker
sample_speaker = speakers[0]

speaker_path = os.path.join(
    TRAIN_DIR,
    sample_speaker
)

print("\n========== SAMPLE SPEAKER ==========")
print("Speaker:", sample_speaker)

# Get chapter folders
chapters = sorted([
    item for item in os.listdir(speaker_path)
    if os.path.isdir(
        os.path.join(speaker_path, item)
    )
])

print("Chapter folders found :", len(chapters))
print("Chapters:", chapters[:10])

# Select first chapter
sample_chapter = chapters[0]

chapter_path = os.path.join(
    speaker_path,
    sample_chapter
)

print("\n========== SAMPLE CHAPTER ==========")
print("Chapter:", sample_chapter)

files = sorted(os.listdir(chapter_path))

print("\nFiles:")
for file in files[:20]:
    print("  ", file)

# Count audio and transcript files
flac_files = [
    f for f in files
    if f.endswith(".flac")
]

transcript_files = [
    f for f in files
    if f.endswith(".trans.txt")
]

print("\n========== FILE CHECK ==========")
print("Audio files      :", len(flac_files))
print("Transcript files :", len(transcript_files))

# Show transcript samples
if transcript_files:

    transcript_path = os.path.join(
        chapter_path,
        transcript_files[0]
    )

    print("\n========== SAMPLE TRANSCRIPTS ==========")

    with open(
        transcript_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f.readlines()[:5]:
            print(line.strip())

    print("\n✓ Audio + transcript structure confirmed.")

else:

    print("\n⚠ No transcript file found.")

print("\n✓ train-clean-100 inspection completed.")

========== TRAIN-CLEAN-100 INSPECTION ==========
Path: /root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/train-clean-100
Speaker folders found : 251
First 10 speakers     : ['103', '1034', '1040', '1069', '1081', '1088', '1098', '1116', '118', '1183']

========== SAMPLE SPEAKER ==========
Speaker: 103
Chapter folders found : 2
Chapters: ['1240', '1241']

========== SAMPLE CHAPTER ==========
Chapter: 1240

Files:
   103-1240-0000.flac
   103-1240-0001.flac
   103-1240-0002.flac
   103-1240-0003.flac
   103-1240-0004.flac
   103-1240-0005.flac
   103-1240-0006.flac
   103-1240-0007.flac
   103-1240-0008.flac
   103-1240-0009.flac
   103-1240-0010.flac
   103-1240-0011.flac
   103-1240-0012.flac
   103-1240-0013.flac
   103-1240-0014.flac
   103-1240-0015.flac
   103-1240-0016.flac
   103-1240-0017.flac
   103-1240-0018.flac
   103-1240-0019.flac

========== FILE CHECK ==========
Audio files      : 58
Transcript files : 1

========== SAMPLE TRANSCRIPTS ==

In [18]:
# ============================================================
# CELL 13 - CREATE ENGLISH ASR TRAINING METADATA
# ============================================================

import os
import glob
import pandas as pd

MAX_SAMPLES = 1000

TRAIN_DIR = os.path.join(
    LIBRISPEECH_DIR,
    "train-clean-100"
)

OUTPUT_DIR = "/content/eldercare_kaggle_data/processed"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("========== CREATING ENGLISH ASR METADATA ==========")
print("Maximum samples :", MAX_SAMPLES)

records = []

# Find all transcript files
transcript_files = glob.glob(
    os.path.join(
        TRAIN_DIR,
        "*",
        "*",
        "*.trans.txt"
    )
)

print("Transcript files found :", len(transcript_files))

for transcript_file in sorted(transcript_files):

    chapter_dir = os.path.dirname(
        transcript_file
    )

    # Extract speaker and chapter IDs
    chapter_name = os.path.basename(
        chapter_dir
    )

    speaker_dir = os.path.dirname(
        chapter_dir
    )

    speaker_id = os.path.basename(
        speaker_dir
    )

    chapter_id = chapter_name

    # Read transcript
    with open(
        transcript_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split(
                " ",
                1
            )

            if len(parts) != 2:
                continue

            audio_id, transcript = parts

            audio_path = os.path.join(
                chapter_dir,
                audio_id + ".flac"
            )

            if not os.path.exists(
                audio_path
            ):
                continue

            records.append({
                "audio": audio_path,
                "text": transcript.lower(),
                "language": "English",
                "speaker_id": speaker_id,
                "chapter_id": chapter_id
            })

            if len(records) >= MAX_SAMPLES:
                break

    if len(records) >= MAX_SAMPLES:
        break


english_df = pd.DataFrame(records)

metadata_path = os.path.join(
    OUTPUT_DIR,
    "english_asr_1000.csv"
)

english_df.to_csv(
    metadata_path,
    index=False
)

print("\n========== DATASET CREATED ==========")
print("Samples collected :", len(english_df))
print("Metadata file     :", metadata_path)

print("\n========== COLUMNS ==========")
print(list(english_df.columns))

print("\n========== SAMPLE RECORDS ==========")
print(
    english_df.head(5).to_string(
        index=False
    )
)

print("\n✓ English ASR training metadata created.")

========== CREATING ENGLISH ASR METADATA ==========
Maximum samples : 1000
Transcript files found : 585

========== DATASET CREATED ==========
Samples collected : 1000
Metadata file     : /content/eldercare_kaggle_data/processed/english_asr_1000.csv

========== COLUMNS ==========
['audio', 'text', 'language', 'speaker_id', 'chapter_id']

========== SAMPLE RECORDS ==========
                                                                                                                          audio                                                                                                                                                                                                                                                                                        text language speaker_id chapter_id
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/train-clean-100/103/1240/103-1240-0000.flac                                                     

In [19]:
# ============================================================
# CELL 14 - VALIDATE ENGLISH ASR AUDIO FILES
# ============================================================

import os
import soundfile as sf
import numpy as np

print("========== VALIDATING ENGLISH AUDIO ==========")

valid_samples = []
invalid_samples = []

sample_rates = []
durations = []

for index, row in english_df.iterrows():

    audio_path = row["audio"]

    try:

        if not os.path.exists(audio_path):
            invalid_samples.append(
                (audio_path, "File not found")
            )
            continue

        info = sf.info(audio_path)

        sample_rate = info.samplerate
        frames = info.frames

        if frames == 0:
            invalid_samples.append(
                (audio_path, "Empty audio")
            )
            continue

        duration = frames / sample_rate

        if duration <= 0:
            invalid_samples.append(
                (audio_path, "Invalid duration")
            )
            continue

        valid_samples.append(audio_path)

        sample_rates.append(sample_rate)
        durations.append(duration)

    except Exception as e:

        invalid_samples.append(
            (audio_path, str(e))
        )

print("\n========== VALIDATION RESULTS ==========")

print("Total samples       :", len(english_df))
print("Valid audio files   :", len(valid_samples))
print("Invalid audio files :", len(invalid_samples))

if sample_rates:

    unique_rates = sorted(
        set(sample_rates)
    )

    print("\n========== AUDIO INFORMATION ==========")

    print("Sample rates found  :", unique_rates)

    print(
        "Minimum duration    : "
        f"{min(durations):.2f} seconds"
    )

    print(
        "Maximum duration    : "
        f"{max(durations):.2f} seconds"
    )

    print(
        "Average duration    : "
        f"{np.mean(durations):.2f} seconds"
    )

    print(
        "Total audio hours   : "
        f"{sum(durations) / 3600:.2f} hours"
    )

if invalid_samples:

    print("\n========== INVALID SAMPLE EXAMPLES ==========")

    for item in invalid_samples[:5]:
        print(item)

else:

    print("\n✓ No invalid audio files detected.")

print("\nAudio validation completed.")

========== VALIDATING ENGLISH AUDIO ==========

========== VALIDATION RESULTS ==========
Total samples       : 1000
Valid audio files   : 1000
Invalid audio files : 0

========== AUDIO INFORMATION ==========
Sample rates found  : [16000]
Minimum duration    : 1.41 seconds
Maximum duration    : 17.24 seconds
Average duration    : 12.11 seconds
Total audio hours   : 3.36 hours

✓ No invalid audio files detected.

Audio validation completed.


In [20]:
# ============================================================
# CELL 15 - CREATE TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

print("========== CREATING TRAIN / VALIDATION SPLIT ==========")

# Make a copy so the original metadata remains unchanged
english_data = english_df.copy()

# Remove accidental empty transcripts
english_data["text"] = english_data["text"].astype(str).str.strip()

english_data = english_data[
    english_data["text"].str.len() > 0
].reset_index(drop=True)

print("Usable samples :", len(english_data))

# ------------------------------------------------------------
# Split data
# ------------------------------------------------------------

try:

    train_df, val_df = train_test_split(
        english_data,
        test_size=0.10,
        random_state=42,
        shuffle=True,
        stratify=english_data["speaker_id"]
    )

    print("Stratified split by speaker: SUCCESS")

except ValueError:

    # Fallback in case a speaker has too few samples
    print(
        "Stratified split was not possible. "
        "Using random split instead."
    )

    train_df, val_df = train_test_split(
        english_data,
        test_size=0.10,
        random_state=42,
        shuffle=True
    )

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\n========== SPLIT RESULTS ==========")
print("Training samples   :", len(train_df))
print("Validation samples :", len(val_df))

print("\n========== TRAINING DATA ==========")
print(train_df.head(3).to_string(index=False))

print("\n========== VALIDATION DATA ==========")
print(val_df.head(3).to_string(index=False))

# ------------------------------------------------------------
# Save split metadata
# ------------------------------------------------------------

TRAIN_METADATA = os.path.join(
    OUTPUT_DIR,
    "english_train.csv"
)

VAL_METADATA = os.path.join(
    OUTPUT_DIR,
    "english_validation.csv"
)

train_df.to_csv(
    TRAIN_METADATA,
    index=False
)

val_df.to_csv(
    VAL_METADATA,
    index=False
)

print("\n========== FILES SAVED ==========")
print("Training metadata   :", TRAIN_METADATA)
print("Validation metadata :", VAL_METADATA)

print("\n✓ Train/validation split completed successfully.")

========== CREATING TRAIN / VALIDATION SPLIT ==========
Usable samples : 1000
Stratified split by speaker: SUCCESS

========== SPLIT RESULTS ==========
Training samples   : 900
Validation samples : 100

========== TRAINING DATA ==========
                                                                                                                                audio                                                                                                                                                                                                                                             text language speaker_id chapter_id
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/train-clean-100/1116/132847/1116-132847-0031.flac                                                                                                                                                                                     to save you repeated bellah but who are y

In [21]:
# ============================================================
# CELL 16 - TEST MMS AUDIO PREPROCESSING
# ============================================================

import torch
import soundfile as sf
import numpy as np

print("========== TESTING MMS AUDIO PREPROCESSING ==========")

# Select first training sample
sample = train_df.iloc[0]

audio_path = sample["audio"]
transcript = sample["text"]

print("Audio file :", audio_path)
print("Transcript :", transcript)

# ------------------------------------------------------------
# Load audio
# ------------------------------------------------------------

audio, sample_rate = sf.read(
    audio_path,
    dtype="float32"
)

print("\n========== ORIGINAL AUDIO ==========")
print("Sample rate :", sample_rate)
print("Shape       :", audio.shape)
print("Duration    :", f"{len(audio) / sample_rate:.2f} seconds")
print("Data type   :", audio.dtype)

# ------------------------------------------------------------
# Convert stereo → mono if necessary
# ------------------------------------------------------------

if audio.ndim > 1:
    audio = np.mean(
        audio,
        axis=1
    )

print("\nAfter mono conversion:")
print("Shape :", audio.shape)

# ------------------------------------------------------------
# Resample if required
# ------------------------------------------------------------

required_rate = processor.feature_extractor.sampling_rate

if sample_rate != required_rate:

    import librosa

    audio = librosa.resample(
        audio,
        orig_sr=sample_rate,
        target_sr=required_rate
    )

    sample_rate = required_rate

print("\n========== FINAL AUDIO ==========")
print("Final sample rate :", sample_rate)
print("Final shape       :", audio.shape)
print(
    "Final duration     : "
    f"{len(audio) / sample_rate:.2f} seconds"
)

# ------------------------------------------------------------
# Pass audio through MMS processor
# ------------------------------------------------------------

inputs = processor(
    audio,
    sampling_rate=sample_rate,
    return_tensors="pt"
)

print("\n========== MMS PROCESSOR OUTPUT ==========")

print("Input keys :", inputs.keys())

if "input_values" in inputs:
    print(
        "input_values shape :",
        inputs["input_values"].shape
    )

# ------------------------------------------------------------
# Move tensor to GPU
# ------------------------------------------------------------

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
    if torch.is_tensor(value)
}

print("\n========== DEVICE CHECK ==========")

for key, value in inputs.items():
    print(
        f"{key:20} : "
        f"shape={tuple(value.shape)}, "
        f"device={value.device}"
    )

print("\n✓ Single-audio MMS preprocessing test completed.")

========== TESTING MMS AUDIO PREPROCESSING ==========
Audio file : /root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/train-clean-100/1116/132847/1116-132847-0031.flac
Transcript : to save you repeated bellah but who are you my little friend

========== ORIGINAL AUDIO ==========
Sample rate : 16000
Shape       : (54000,)
Duration    : 3.38 seconds
Data type   : float32

After mono conversion:
Shape : (54000,)

========== FINAL AUDIO ==========
Final sample rate : 16000
Final shape       : (54000,)
Final duration     : 3.38 seconds

========== MMS PROCESSOR OUTPUT ==========
Input keys : KeysView({'input_values': tensor([[ 0.0028,  0.0020,  0.0012,  ..., -0.0043, -0.0020, -0.0028]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32)})
input_values shape : torch.Size([1, 54000])

========== DEVICE CHECK ==========
input_values         : shape=(1, 54000), device=cuda:0
attention_mask       : shape=(1, 54000), device=cuda:0

✓ Single-a

In [22]:
# ============================================================
# CELL 17 - CREATE HUGGING FACE ASR DATASETS
# ============================================================

from datasets import Dataset, Audio

print("========== CREATING HUGGING FACE ASR DATASETS ==========")

# ------------------------------------------------------------
# Select only the columns required for ASR
# ------------------------------------------------------------

train_dataset_df = train_df[
    ["audio", "text", "language"]
].copy()

val_dataset_df = val_df[
    ["audio", "text", "language"]
].copy()

# ------------------------------------------------------------
# Convert pandas → Hugging Face Dataset
# ------------------------------------------------------------

train_dataset = Dataset.from_pandas(
    train_dataset_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_dataset_df,
    preserve_index=False
)

# ------------------------------------------------------------
# Cast audio column
# MMS expects 16 kHz audio.
# This keeps audio loading lazy instead of loading
# all 900 files into memory immediately.
# ------------------------------------------------------------

train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

val_dataset = val_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("\n========== DATASET INFORMATION ==========")

print("Training dataset   :", train_dataset)
print("Validation dataset :", val_dataset)

print("\nTraining samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))

print("\nColumns:")
print(train_dataset.column_names)

# ------------------------------------------------------------
# Inspect one training example
# ------------------------------------------------------------

print("\n========== SAMPLE TRAINING RECORD ==========")

sample = train_dataset[0]

print("Audio information:")
print(
    "  Array shape     :",
    sample["audio"]["array"].shape
)

print(
    "  Sampling rate   :",
    sample["audio"]["sampling_rate"]
)

print(
    "  Duration        :",
    f"{len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f} seconds"
)

print("\nTranscript:")
print(" ", sample["text"])

print("\nLanguage:")
print(" ", sample["language"])

# ------------------------------------------------------------
# Inspect validation example
# ------------------------------------------------------------

print("\n========== SAMPLE VALIDATION RECORD ==========")

val_sample = val_dataset[0]

print(
    "Validation audio shape :",
    val_sample["audio"]["array"].shape
)

print(
    "Validation sample rate :",
    val_sample["audio"]["sampling_rate"]
)

print(
    "Validation transcript  :",
    val_sample["text"]
)

print("\n✓ Hugging Face ASR datasets created successfully.")

========== CREATING HUGGING FACE ASR DATASETS ==========

========== DATASET INFORMATION ==========
Training dataset   : Dataset({
    features: ['audio', 'text', 'language'],
    num_rows: 900
})
Validation dataset : Dataset({
    features: ['audio', 'text', 'language'],
    num_rows: 100
})

Training samples   : 900
Validation samples : 100

Columns:
['audio', 'text', 'language']

========== SAMPLE TRAINING RECORD ==========
Audio information:
  Array shape     : (54000,)
  Sampling rate   : 16000
  Duration        : 3.38 seconds

Transcript:
  to save you repeated bellah but who are you my little friend

Language:
  English

========== SAMPLE VALIDATION RECORD ==========
Validation audio shape : (190720,)
Validation sample rate : 16000
Validation transcript  : since the law won't hold me at your convenience in smithtown i keep myself at your service here if bobby permits it could you ask more bobby shrank from the man with whom he had idled away so much time and money

✓ Hugging Fac

In [23]:
# ============================================================
# CELL 18 - TEST MMS TRANSCRIPT → CTC LABEL CONVERSION
# Transformers 4.57.6 compatible
# ============================================================

import torch

print("========== TESTING MMS LABEL PROCESSING ==========")

# ------------------------------------------------------------
# Activate English MMS adapter
# ------------------------------------------------------------

processor.tokenizer.set_target_lang("eng")
model.load_adapter("eng")

print("Active MMS language : English")
print("Language code       : eng")

# ------------------------------------------------------------
# Get one transcript
# ------------------------------------------------------------

sample_text = train_dataset[0]["text"]

print("\n========== SAMPLE TRANSCRIPT ==========")
print("Text:")
print(sample_text)

# ------------------------------------------------------------
# Convert transcript to CTC token IDs
# ------------------------------------------------------------
# IMPORTANT:
# Transformers 4.57.6 supports processing labels directly
# through the tokenizer.

labels = processor.tokenizer(
    sample_text,
    return_tensors="pt"
).input_ids

print("\n========== LABEL INFORMATION ==========")

print("Label tensor shape :", labels.shape)
print("Label data type    :", labels.dtype)
print("Label device       :", labels.device)

print("\nToken IDs:")
print(labels[0].tolist())

print("\nNumber of labels   :", labels.shape[1])

# ------------------------------------------------------------
# Decode labels back to text
# ------------------------------------------------------------

decoded_text = processor.tokenizer.decode(
    labels[0],
    skip_special_tokens=True
)

print("\n========== LABEL DECODING TEST ==========")

print("Original text:")
print(sample_text)

print("\nDecoded text:")
print(decoded_text)

# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

if decoded_text.strip().lower() == sample_text.strip().lower():

    print("\n✓ Transcript → labels → transcript conversion PASSED.")

else:

    print("\n⚠ Decoded text differs slightly from original.")

print("\n✓ MMS CTC label processing test completed.")

========== TESTING MMS LABEL PROCESSING ==========
Active MMS language : English
Language code       : eng

========== SAMPLE TRANSCRIPT ==========
Text:
to save you repeated bellah but who are you my little friend

========== LABEL INFORMATION ==========
Label tensor shape : torch.Size([1, 60])
Label data type    : torch.int64
Label device       : cpu

Token IDs:
[6, 8, 4, 11, 7, 25, 5, 4, 22, 8, 17, 4, 12, 5, 20, 5, 7, 6, 5, 15, 4, 24, 5, 14, 14, 7, 13, 4, 24, 17, 6, 4, 23, 13, 8, 4, 7, 12, 5, 4, 22, 8, 17, 4, 18, 22, 4, 14, 9, 6, 6, 14, 5, 4, 19, 12, 9, 5, 10, 15]

Number of labels   : 60

========== LABEL DECODING TEST ==========
Original text:
to save you repeated bellah but who are you my little friend

Decoded text:
to save you repeated belah but who are you my litle friend

⚠ Decoded text differs slightly from original.

✓ MMS CTC label processing test completed.


In [24]:
# ============================================================
# CELL 19 - CREATE MMS ASR PREPROCESSING FUNCTION
# Transformers 4.57.6 compatible
# ============================================================

import numpy as np
import torch

print("========== CREATING MMS PREPROCESSING FUNCTION ==========")

# ------------------------------------------------------------
# Activate English MMS adapter
# ------------------------------------------------------------

processor.tokenizer.set_target_lang("eng")
model.load_adapter("eng")

print("Active language : English")
print("Language code   : eng")

# ------------------------------------------------------------
# Preprocessing function
# ------------------------------------------------------------

def prepare_example(example):

    # --------------------------------------------------------
    # Get audio
    # --------------------------------------------------------

    audio = example["audio"]

    audio_array = np.asarray(
        audio["array"],
        dtype=np.float32
    )

    sampling_rate = audio["sampling_rate"]

    # --------------------------------------------------------
    # Process AUDIO using feature extractor only
    # --------------------------------------------------------
    # This avoids the tokenizer/audio routing issue in
    # Transformers 4.57.6.

    audio_inputs = processor.feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_attention_mask=True
    )

    # --------------------------------------------------------
    # Process TRANSCRIPT using tokenizer only
    # --------------------------------------------------------

    label_ids = processor.tokenizer(
        example["text"]
    ).input_ids

    # --------------------------------------------------------
    # Return model-ready example
    # --------------------------------------------------------

    return {
        "input_values": audio_inputs["input_values"][0],
        "attention_mask": audio_inputs["attention_mask"][0],
        "labels": label_ids
    }


print("✓ Preprocessing function created.")

# ============================================================
# TEST ONE TRAINING SAMPLE
# ============================================================

print("\n========== TESTING ON ONE SAMPLE ==========")

test_example = train_dataset[0]

processed_example = prepare_example(
    test_example
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nProcessed fields:")
print(
    list(processed_example.keys())
)

print("\nInput values:")
print(
    "  Shape :",
    processed_example["input_values"].shape
)

print(
    "  Type  :",
    type(processed_example["input_values"])
)

print("\nAttention mask:")
print(
    "  Shape :",
    processed_example["attention_mask"].shape
)

print(
    "  Type  :",
    type(processed_example["attention_mask"])
)

print("\nLabels:")
print(
    "  Length :",
    len(processed_example["labels"])
)

print(
    "  First IDs :",
    processed_example["labels"][:20]
)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert len(processed_example["input_values"]) > 0
assert len(processed_example["attention_mask"]) > 0
assert len(processed_example["labels"]) > 0

print("\n✓ Audio → input_values conversion PASSED.")
print("✓ Audio → attention_mask conversion PASSED.")
print("✓ Transcript → labels conversion PASSED.")
print("✓ Single-example preprocessing PASSED.")

print("\nMMS preprocessing function is ready.")

========== CREATING MMS PREPROCESSING FUNCTION ==========
Active language : English
Language code   : eng
✓ Preprocessing function created.

========== TESTING ON ONE SAMPLE ==========

Processed fields:
['input_values', 'attention_mask', 'labels']

Input values:
  Shape : (54000,)
  Type  : <class 'numpy.ndarray'>

Attention mask:
  Shape : (54000,)
  Type  : <class 'numpy.ndarray'>

Labels:
  Length : 60
  First IDs : [6, 8, 4, 11, 7, 25, 5, 4, 22, 8, 17, 4, 12, 5, 20, 5, 7, 6, 5, 15]

✓ Audio → input_values conversion PASSED.
✓ Audio → attention_mask conversion PASSED.
✓ Transcript → labels conversion PASSED.
✓ Single-example preprocessing PASSED.

MMS preprocessing function is ready.


In [25]:
# ============================================================
# CELL 26A - REBUILD PROCESSED DATASETS
# ============================================================

import os
import pandas as pd
import numpy as np
from datasets import Dataset, Audio

print("========== VERIFYING PROCESSED DATASET ==========\n")

# ------------------------------------------------------------
# Paths to previously saved CSV files
# ------------------------------------------------------------

TRAIN_CSV = "/content/eldercare_kaggle_data/processed/english_train.csv"
VAL_CSV = "/content/eldercare_kaggle_data/processed/english_validation.csv"

print("Train CSV :", TRAIN_CSV)
print("Val CSV   :", VAL_CSV)

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

if not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(
        f"Training CSV not found:\n{TRAIN_CSV}\n"
        "The English dataset may need to be processed again."
    )

if not os.path.exists(VAL_CSV):
    raise FileNotFoundError(
        f"Validation CSV not found:\n{VAL_CSV}\n"
        "The English dataset may need to be processed again."
    )

print("\n✓ Training CSV found")
print("✓ Validation CSV found")

# ------------------------------------------------------------
# Load CSV files
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("\n========== CSV INFORMATION ==========")

print("Training rows   :", len(train_df))
print("Validation rows :", len(val_df))

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

# ------------------------------------------------------------
# Create Hugging Face datasets
# ------------------------------------------------------------

train_dataset_df = train_df[["audio", "text", "language"]].copy()
val_dataset_df = val_df[["audio", "text", "language"]].copy()

train_dataset = Dataset.from_pandas(
    train_dataset_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_dataset_df,
    preserve_index=False
)

# ------------------------------------------------------------
# Convert audio column
# ------------------------------------------------------------

train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

val_dataset = val_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("\n✓ Hugging Face training dataset recreated")
print("✓ Hugging Face validation dataset recreated")

# ------------------------------------------------------------
# Make sure English MMS adapter is active
# ------------------------------------------------------------

processor.tokenizer.set_target_lang("eng")
model.load_adapter("eng")

print("✓ English MMS adapter activated")

# ------------------------------------------------------------
# Preprocessing function
# ------------------------------------------------------------

def prepare_example(example):

    audio = example["audio"]

    audio_array = np.asarray(
        audio["array"],
        dtype=np.float32
    )

    sampling_rate = audio["sampling_rate"]

    # Audio preprocessing
    audio_inputs = processor.feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_attention_mask=True
    )

    # Text → token IDs
    text = str(example["text"]).strip()

    label_ids = processor.tokenizer(text).input_ids

    return {
        "input_values": audio_inputs["input_values"][0],
        "attention_mask": audio_inputs["attention_mask"][0],
        "labels": label_ids
    }

# ------------------------------------------------------------
# Process training dataset
# ------------------------------------------------------------

print("\n========== PROCESSING TRAINING DATA ==========")

train_processed = train_dataset.map(
    prepare_example,
    remove_columns=train_dataset.column_names,
    desc="Processing English training audio"
)

print("✓ Training dataset processed")

# ------------------------------------------------------------
# Process validation dataset
# ------------------------------------------------------------

print("\n========== PROCESSING VALIDATION DATA ==========")

val_processed = val_dataset.map(
    prepare_example,
    remove_columns=val_dataset.column_names,
    desc="Processing English validation audio"
)

print("✓ Validation dataset processed")

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n========== FINAL DATASET VERIFICATION ==========")

print("Training samples       :", len(train_processed))
print("Validation samples     :", len(val_processed))

print("\nTraining features:")
print(train_processed.features)

print("\nValidation features:")
print(val_processed.features)

# ------------------------------------------------------------
# Verify first sample
# ------------------------------------------------------------

sample = train_processed[0]

print("\n========== SAMPLE VERIFICATION ==========")

print("Input values type      :", type(sample["input_values"]))
print("Input values length    :", len(sample["input_values"]))

print("Attention mask type    :", type(sample["attention_mask"]))
print("Attention mask length  :", len(sample["attention_mask"]))

print("Labels type            :", type(sample["labels"]))
print("Labels length          :", len(sample["labels"]))

print("\n============================================")
print("✓ train_processed recreated")
print("✓ val_processed recreated")
print("✓ Audio preprocessing completed")
print("✓ Labels recreated")
print("============================================")

print("\nReady to rebuild the Trainer.")

========== VERIFYING PROCESSED DATASET ==========

Train CSV : /content/eldercare_kaggle_data/processed/english_train.csv
Val CSV   : /content/eldercare_kaggle_data/processed/english_validation.csv

✓ Training CSV found
✓ Validation CSV found

========== CSV INFORMATION ==========
Training rows   : 900
Validation rows : 100

Training columns:
['audio', 'text', 'language', 'speaker_id', 'chapter_id']

Validation columns:
['audio', 'text', 'language', 'speaker_id', 'chapter_id']

✓ Hugging Face training dataset recreated
✓ Hugging Face validation dataset recreated
✓ English MMS adapter activated

========== PROCESSING TRAINING DATA ==========


Processing English training audio:   0%|          | 0/900 [00:00<?, ? examples/s]

✓ Training dataset processed

========== PROCESSING VALIDATION DATA ==========


Processing English validation audio:   0%|          | 0/100 [00:00<?, ? examples/s]

✓ Validation dataset processed

========== FINAL DATASET VERIFICATION ==========
Training samples       : 900
Validation samples     : 100

Training features:
{'input_values': List(Value('float32')), 'attention_mask': List(Value('int32')), 'labels': List(Value('int64'))}

Validation features:
{'input_values': List(Value('float32')), 'attention_mask': List(Value('int32')), 'labels': List(Value('int64'))}

========== SAMPLE VERIFICATION ==========
Input values type      : <class 'list'>
Input values length    : 54000
Attention mask type    : <class 'list'>
Attention mask length  : 54000
Labels type            : <class 'list'>
Labels length          : 60

✓ train_processed recreated
✓ val_processed recreated
✓ Audio preprocessing completed
✓ Labels recreated

Ready to rebuild the Trainer.


In [26]:
# ============================================================
# CELL 21 - CTC DYNAMIC PADDING DATA COLLATOR
# ============================================================

import torch
from dataclasses import dataclass
from typing import Dict, List, Union

print("========== CREATING CTC DATA COLLATOR ==========")

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ):

        # ----------------------------------------------------
        # Separate audio inputs from labels
        # ----------------------------------------------------

        input_features = [
            {
                "input_values": feature["input_values"],
                "attention_mask": feature["attention_mask"]
            }
            for feature in features
        ]

        label_features = [
            {
                "input_ids": feature["labels"]
            }
            for feature in features
        ]

        # ----------------------------------------------------
        # Dynamic padding for audio
        # ----------------------------------------------------

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # ----------------------------------------------------
        # Dynamic padding for CTC labels
        # ----------------------------------------------------

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # ----------------------------------------------------
        # Replace padding tokens with -100
        # ----------------------------------------------------
        # CTC loss ignores positions containing -100.

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        batch["labels"] = labels

        # ----------------------------------------------------
        # Ensure memory-efficient float32 audio
        # ----------------------------------------------------

        batch["input_values"] = batch["input_values"].float()

        return batch


# ------------------------------------------------------------
# Create collator
# ------------------------------------------------------------

data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("✓ CTC data collator created.")
print("✓ Dynamic audio padding enabled.")
print("✓ Dynamic label padding enabled.")
print("✓ Label padding tokens converted to -100.")
print("✓ Audio converted to float32.")
print("\nCTC data collator is ready.")

========== CREATING CTC DATA COLLATOR ==========
✓ CTC data collator created.
✓ Dynamic audio padding enabled.
✓ Dynamic label padding enabled.
✓ Label padding tokens converted to -100.
✓ Audio converted to float32.

CTC data collator is ready.


In [27]:
# ============================================================
# CELL 22 - TEST CTC DATA COLLATOR
# ============================================================

print("========== TESTING CTC DATA COLLATOR ==========")

# ------------------------------------------------------------
# Take 4 samples from training dataset
# ------------------------------------------------------------

test_features = [
    train_processed[i]
    for i in range(4)
]

print("\nTest batch size:", len(test_features))

# ------------------------------------------------------------
# Apply collator
# ------------------------------------------------------------

test_batch = data_collator(test_features)

# ------------------------------------------------------------
# Display batch information
# ------------------------------------------------------------

print("\n========== BATCH INFORMATION ==========")

print("Batch keys:")
print(list(test_batch.keys()))

print("\nInput values:")
print("  Shape :", tuple(test_batch["input_values"].shape))
print("  Dtype :", test_batch["input_values"].dtype)
print("  Device:", test_batch["input_values"].device)

print("\nAttention mask:")
print("  Shape :", tuple(test_batch["attention_mask"].shape))
print("  Dtype :", test_batch["attention_mask"].dtype)

print("\nLabels:")
print("  Shape :", tuple(test_batch["labels"].shape))
print("  Dtype :", test_batch["labels"].dtype)

# ------------------------------------------------------------
# Check padding
# ------------------------------------------------------------

padding_count = (
    test_batch["labels"] == -100
).sum().item()

print("\nNumber of -100 label padding positions:", padding_count)

# ------------------------------------------------------------
# Validate tensors
# ------------------------------------------------------------

assert "input_values" in test_batch
assert "attention_mask" in test_batch
assert "labels" in test_batch

assert test_batch["input_values"].ndim == 2
assert test_batch["attention_mask"].ndim == 2
assert test_batch["labels"].ndim == 2

assert test_batch["input_values"].dtype == torch.float32
assert test_batch["labels"].dtype == torch.int64

assert test_batch["input_values"].shape[0] == 4
assert test_batch["attention_mask"].shape[0] == 4
assert test_batch["labels"].shape[0] == 4

print("\n============================================")
print("✓ Batch creation              : PASSED")
print("✓ Dynamic audio padding       : PASSED")
print("✓ Dynamic label padding       : PASSED")
print("✓ Float32 audio tensors       : PASSED")
print("✓ CTC -100 label masking      : PASSED")
print("✓ Tensor dimensions           : PASSED")
print("============================================")

print("\nCTC data collator test PASSED.")
print("Ready for the training configuration.")

========== TESTING CTC DATA COLLATOR ==========

Test batch size: 4

========== BATCH INFORMATION ==========
Batch keys:
['input_values', 'attention_mask', 'labels']

Input values:
  Shape : (4, 232960)
  Dtype : torch.float32
  Device: cpu

Attention mask:
  Shape : (4, 232960)
  Dtype : torch.int64

Labels:
  Shape : (4, 244)
  Dtype : torch.int64

Number of -100 label padding positions: 212

✓ Batch creation              : PASSED
✓ Dynamic audio padding       : PASSED
✓ Dynamic label padding       : PASSED
✓ Float32 audio tensors       : PASSED
✓ CTC -100 label masking      : PASSED
✓ Tensor dimensions           : PASSED

CTC data collator test PASSED.
Ready for the training configuration.


In [28]:
# ============================================================
# CELL 23 - CONFIGURE MMS FOR T4 TRAINING
# ============================================================

import torch

print("========== CONFIGURING MMS FOR TRAINING ==========")

# ------------------------------------------------------------
# Activate English adapter
# ------------------------------------------------------------

processor.tokenizer.set_target_lang("eng")
model.load_adapter("eng")

print("Active language : English")
print("Language code   : eng")

# ------------------------------------------------------------
# Freeze the large MMS backbone
# ------------------------------------------------------------

for param in model.wav2vec2.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Keep the CTC output layer trainable
# ------------------------------------------------------------

for param in model.lm_head.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Enable gradient checkpointing
# ------------------------------------------------------------

model.gradient_checkpointing_enable()

# Required when using gradient checkpointing
model.config.use_cache = False

# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_parameters = total_parameters - trainable_parameters

print("\n========== PARAMETER INFORMATION ==========")

print(
    f"Total parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters  : "
    f"{trainable_parameters:,}"
)

print(
    f"Frozen parameters     : "
    f"{frozen_parameters:,}"
)

print(
    f"Trainable percentage  : "
    f"{100 * trainable_parameters / total_parameters:.4f}%"
)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 1e-4
NUM_EPOCHS = 3

print("\n========== TRAINING SETTINGS ==========")

print("Train batch size              :", TRAIN_BATCH_SIZE)
print("Evaluation batch size         :", EVAL_BATCH_SIZE)
print("Gradient accumulation steps   :", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size          :",
      TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate                 :", LEARNING_RATE)
print("Epochs                        :", NUM_EPOCHS)
print("Gradient checkpointing        :", True)

# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print("\n========== GPU ==========")
    print("GPU                          :", gpu_name)
    print(f"GPU memory                   : {gpu_memory:.2f} GB")

else:

    print("\n⚠ CUDA GPU is not available.")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n============================================")
print("✓ English MMS adapter activated")
print("✓ MMS backbone frozen")
print("✓ CTC head trainable")
print("✓ Gradient checkpointing enabled")
print("✓ T4-safe batch configuration created")
print("============================================")

print("\nMMS training configuration is ready.")

========== CONFIGURING MMS FOR TRAINING ==========
Active language : English
Language code   : eng

========== PARAMETER INFORMATION ==========
Total parameters      : 964,845,850
Trainable parameters  : 197,274
Frozen parameters     : 964,648,576
Trainable percentage  : 0.0204%

========== TRAINING SETTINGS ==========
Train batch size              : 2
Evaluation batch size         : 2
Gradient accumulation steps   : 4
Effective batch size          : 8
Learning rate                 : 0.0001
Epochs                        : 3
Gradient checkpointing        : True

========== GPU ==========
GPU                          : Tesla T4
GPU memory                   : 14.56 GB

✓ English MMS adapter activated
✓ MMS backbone frozen
✓ CTC head trainable
✓ Gradient checkpointing enabled
✓ T4-safe batch configuration created

MMS training configuration is ready.


In [29]:
# ============================================================
# CELL 24 - VERIFY TRAINABLE PARAMETERS
# ============================================================

print("========== VERIFYING TRAINABLE PARAMETERS ==========")

trainable_names = []
trainable_count = 0

for name, param in model.named_parameters():

    if param.requires_grad:
        trainable_names.append(name)
        trainable_count += param.numel()

print("\nTrainable parameter tensors:")
print("--------------------------------------------")

for name in trainable_names:
    print(name)

print("--------------------------------------------")

print("\nTrainable tensors :", len(trainable_names))
print("Trainable parameters:", f"{trainable_count:,}")

# ------------------------------------------------------------
# Verify backbone is frozen
# ------------------------------------------------------------

backbone_trainable = []

for name, param in model.wav2vec2.named_parameters():

    if param.requires_grad:
        backbone_trainable.append(name)

print("\nBackbone trainable tensors:", len(backbone_trainable))

# ------------------------------------------------------------
# Verify CTC head
# ------------------------------------------------------------

lm_head_trainable = []

for name, param in model.lm_head.named_parameters():

    if param.requires_grad:
        lm_head_trainable.append(name)

print("CTC head trainable tensors:", len(lm_head_trainable))

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert trainable_count == 197274
assert len(backbone_trainable) == 0
assert len(lm_head_trainable) > 0

print("\n============================================")
print("✓ Trainable parameter verification PASSED")
print("✓ MMS backbone is completely frozen")
print("✓ CTC head is trainable")
print("✓ Parameter count matches Cell 23")
print("============================================")

print("\nReady to create the training configuration.")

========== VERIFYING TRAINABLE PARAMETERS ==========

Trainable parameter tensors:
--------------------------------------------
lm_head.weight
lm_head.bias
--------------------------------------------

Trainable tensors : 2
Trainable parameters: 197,274

Backbone trainable tensors: 0
CTC head trainable tensors: 2

✓ Trainable parameter verification PASSED
✓ MMS backbone is completely frozen
✓ CTC head is trainable
✓ Parameter count matches Cell 23

Ready to create the training configuration.


In [34]:
# ============================================================
# FIX - CLEAN REINSTALL OF TRANSFORMERS
# ============================================================

import sys

print("========== FIXING TRANSFORMERS INSTALLATION ==========")
print("Python version:", sys.version)

# Remove broken/inconsistent installations
!pip uninstall -y transformers tokenizers -q

# Install clean compatible versions
!pip install -q --no-cache-dir \
    "transformers==4.57.6" \
    "tokenizers==0.22.2"

print("\n========================================================")
print("✓ Transformers reinstalled successfully")
print("✓ Tokenizers reinstalled successfully")
print("========================================================")
print("\n⚠️ IMPORTANT:")
print("Restart the Colab runtime now.")
print("Then DO NOT run Cell 25 immediately.")
print("Come back and send me the output after restarting.")

========== FIXING TRANSFORMERS INSTALLATION ==========
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.1 MB/s eta 0:00:00

✓ Transformers reinstalled successfully
✓ Tokenizers reinstalled successfully

⚠️ IMPORTANT:
Restart the Colab runtime now.
Then DO NOT run Cell 25 immediately.
Come back and send me the output after restarting.


In [3]:
# ============================================================
# CELL 26 FIX - FINAL DEPENDENCY VERIFICATION
# ============================================================

import sys

print("========== ELDERCARE TRAINING DEPENDENCY CHECK ==========")
print("Python version:", sys.version)

# ------------------------------------------------------------
# IMPORT REQUIRED PACKAGES
# ------------------------------------------------------------

import numpy as np
import torch
import evaluate
import jiwer
import datasets
import accelerate
import transformers

from transformers import (
    Trainer,
    TrainingArguments,
    AutoProcessor,
    Wav2Vec2ForCTC
)

# ------------------------------------------------------------
# PACKAGE VERSIONS
# ------------------------------------------------------------

print("\n========== PACKAGE VERSIONS ==========")

print("NumPy        :", np.__version__)
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Evaluate     :", evaluate.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)

# JiWER does not reliably expose __version__
print("JiWER        : Installed")

# ------------------------------------------------------------
# TRANSFORMERS IMPORT CHECK
# ------------------------------------------------------------

print("\n========== TRANSFORMERS CHECK ==========")

print("Trainer               : OK")
print("TrainingArguments     : OK")
print("AutoProcessor         : OK")
print("Wav2Vec2ForCTC        : OK")

# ------------------------------------------------------------
# GPU CHECK
# ------------------------------------------------------------

print("\n========== GPU CHECK ==========")

if torch.cuda.is_available():

    print("CUDA available        : True")
    print("GPU                   :", torch.cuda.get_device_name(0))
    print("CUDA version          :", torch.version.cuda)

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        "GPU memory            : "
        f"{gpu_memory:.2f} GB"
    )

else:

    print("CUDA available        : False")
    print("GPU                   : Not available")

# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n========== FINAL STATUS ==========")

print("NumPy                 : ✓")
print("PyTorch               : ✓")
print("Transformers          : ✓")
print("Evaluate              : ✓")
print("JiWER                 : ✓")
print("Datasets              : ✓")
print("Accelerate            : ✓")
print("Trainer               : ✓")
print("TrainingArguments     : ✓")
print("MMS model classes     : ✓")

print("\n==========================================================")
print("✓ ALL REQUIRED TRAINING DEPENDENCIES ARE READY")
print("==========================================================")

========== ELDERCARE TRAINING DEPENDENCY CHECK ==========
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]

========== PACKAGE VERSIONS ==========
NumPy        : 2.1.3
PyTorch      : 2.11.0+cu128
Transformers : 5.16.1
Evaluate     : 0.4.6
Datasets     : 4.0.0
Accelerate   : 1.14.0
JiWER        : Installed

========== TRANSFORMERS CHECK ==========
Trainer               : OK
TrainingArguments     : OK
AutoProcessor         : OK
Wav2Vec2ForCTC        : OK

========== GPU CHECK ==========
CUDA available        : True
GPU                   : Tesla T4
CUDA version          : 12.8
GPU memory            : 14.56 GB

========== FINAL STATUS ==========
NumPy                 : ✓
PyTorch               : ✓
Transformers          : ✓
Evaluate              : ✓
JiWER                 : ✓
Datasets              : ✓
Accelerate            : ✓
Trainer               : ✓
TrainingArguments     : ✓
MMS model classes     : ✓

✓ ALL REQUIRED TRAINING DEPENDENCIES ARE READY


In [6]:
# ============================================================
# CELL 27A - CHECK ELDERCARE DATA AFTER RUNTIME RESTART
# ============================================================

import os

print("========== CHECKING ELDERCARE DATA ==========")

BASE_DIR = "/content/eldercare_kaggle_data"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

print("\nBase directory:")
print(BASE_DIR)
print("Exists:", os.path.exists(BASE_DIR))

print("\nProcessed directory:")
print(PROCESSED_DIR)
print("Exists:", os.path.exists(PROCESSED_DIR))

print("\n========== /content DIRECTORY ==========")

for item in os.listdir("/content"):
    print(item)

print("\n========== SEARCHING FOR CSV FILES ==========")

found_csv = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            found_csv.append(os.path.join(root, file))

if found_csv:
    for path in found_csv:
        print("FOUND:", path)
else:
    print("No CSV files found under /content.")

print("\n================================================")
print("✓ DATA CHECK COMPLETED")
print("================================================")

========== CHECKING ELDERCARE DATA ==========

Base directory:
/content/eldercare_kaggle_data
Exists: False

Processed directory:
/content/eldercare_kaggle_data/processed
Exists: False

========== /content DIRECTORY ==========
.config
sample_data

========== SEARCHING FOR CSV FILES ==========
FOUND: /content/sample_data/california_housing_train.csv
FOUND: /content/sample_data/mnist_test.csv
FOUND: /content/sample_data/california_housing_test.csv
FOUND: /content/sample_data/mnist_train_small.csv

✓ DATA CHECK COMPLETED


In [7]:
# ============================================================
# CELL 27B - CHECK KAGGLEHUB CACHE
# ============================================================

import os
import glob

print("========== CHECKING KAGGLEHUB CACHE ==========")

KAGGLE_CACHE = "/root/.cache/kagglehub"

if os.path.exists(KAGGLE_CACHE):
    print("KaggleHub cache : FOUND")

    # Search specifically for the LibriSpeech dataset
    matches = glob.glob(
        "/root/.cache/kagglehub/datasets/**/librispeech-clean/**",
        recursive=True
    )

    if matches:
        print("\nLibriSpeech dataset cache FOUND:")
        for path in matches:
            print(path)

        print("\n✓ We can reuse the existing dataset.")
        print("✓ No 28 GB download should be required.")

    else:
        print("\nLibriSpeech cache was NOT found.")
        print("We may need to download the dataset again.")

else:
    print("KaggleHub cache : NOT FOUND")
    print("We may need to download the dataset again.")

print("\n================================================")
print("✓ KAGGLEHUB CACHE CHECK COMPLETED")
print("================================================")

========== CHECKING KAGGLEHUB CACHE ==========
KaggleHub cache : NOT FOUND
We may need to download the dataset again.

✓ KAGGLEHUB CACHE CHECK COMPLETED


In [8]:
# ============================================================
# CELL 27C - DOWNLOAD LIBRISPEECH CLEAN DATASET
# ============================================================

import os
import kagglehub

print("========== DOWNLOADING LIBRISPEECH ==========")

DATASET_NAME = "victorling/librispeech-clean"

print("Dataset:", DATASET_NAME)
print("\nStarting download...")
print("⚠️ Dataset is approximately 28 GB.")
print("Please do not stop the Colab runtime while downloading.\n")

english_kaggle_path = kagglehub.dataset_download(DATASET_NAME)

print("\n========== DOWNLOAD COMPLETED ==========")
print("Dataset path:")
print(english_kaggle_path)

print("\nContents:")

for item in os.listdir(english_kaggle_path):
    print(" -", item)

print("\n================================================")
print("✓ LIBRISPEECH DATASET READY")
print("================================================")

========== DOWNLOADING LIBRISPEECH ==========
Dataset: victorling/librispeech-clean

Starting download...
⚠️ Dataset is approximately 28 GB.
Please do not stop the Colab runtime while downloading.



100%|██████████| 28.1G/28.1G [26:45<00:00, 18.8MB/s]

Extracting files...



========== DOWNLOAD COMPLETED ==========
Dataset path:
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1

Contents:
 - LibriSpeech

✓ LIBRISPEECH DATASET READY


In [9]:
# ============================================================
# CELL 27D - RECREATE ENGLISH ASR METADATA
# ============================================================

import os
import pandas as pd

DATASET_ROOT = "/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech"

OUTPUT_DIR = "/content/eldercare_kaggle_data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SAMPLES = 1000

print("========== RECREATING ENGLISH ASR METADATA ==========")
print("Dataset root:")
print(DATASET_ROOT)

# ------------------------------------------------------------
# Check dataset
# ------------------------------------------------------------

if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(
        f"LibriSpeech dataset not found at:\n{DATASET_ROOT}"
    )

print("\nDataset found ✓")

# ------------------------------------------------------------
# Search for LibriSpeech transcript files
# ------------------------------------------------------------

transcript_files = []

for root, dirs, files in os.walk(DATASET_ROOT):
    for file in files:
        if file.endswith(".trans.txt"):
            transcript_files.append(
                os.path.join(root, file)
            )

print("\nTranscript files found:", len(transcript_files))

if len(transcript_files) == 0:
    raise RuntimeError("No LibriSpeech transcript files were found.")

# ------------------------------------------------------------
# Read transcripts
# ------------------------------------------------------------

records = []

for transcript_path in transcript_files:

    chapter_dir = os.path.dirname(transcript_path)

    # Example:
    # 103-1240.trans.txt
    transcript_name = os.path.basename(transcript_path)

    with open(
        transcript_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split(" ", 1)

            if len(parts) != 2:
                continue

            audio_id = parts[0]
            text = parts[1].strip()

            audio_path = os.path.join(
                chapter_dir,
                audio_id + ".flac"
            )

            if not os.path.exists(audio_path):
                continue

            id_parts = audio_id.split("-")

            speaker_id = id_parts[0]
            chapter_id = id_parts[1]

            records.append({
                "audio": audio_path,
                "text": text,
                "language": "English",
                "speaker_id": speaker_id,
                "chapter_id": chapter_id
            })

            if len(records) >= MAX_SAMPLES:
                break

    if len(records) >= MAX_SAMPLES:
        break

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(records)

print("\n========== DATASET SUMMARY ==========")
print("Total samples collected:", len(df))

if len(df) == 0:
    raise RuntimeError("No valid audio samples were collected.")

print("\nSample records:")
print(df.head(5).to_string(index=False))

# ------------------------------------------------------------
# Save complete 1000-sample metadata
# ------------------------------------------------------------

ALL_METADATA = os.path.join(
    OUTPUT_DIR,
    "english_asr_1000.csv"
)

df.to_csv(
    ALL_METADATA,
    index=False
)

print("\nSaved:")
print(ALL_METADATA)

# ------------------------------------------------------------
# Verify audio files
# ------------------------------------------------------------

valid_count = 0
invalid_count = 0

for path in df["audio"]:

    if os.path.exists(path):
        valid_count += 1
    else:
        invalid_count += 1

print("\n========== AUDIO VALIDATION ==========")
print("Valid audio files  :", valid_count)
print("Missing audio files:", invalid_count)

print("\n================================================")
print("✓ ENGLISH ASR METADATA RECREATED")
print("================================================")

========== RECREATING ENGLISH ASR METADATA ==========
Dataset root:
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech

Dataset found ✓

Transcript files found: 2866

========== DATASET SUMMARY ==========
Total samples collected: 1000

Sample records:
                                                                                                                              audio                                                                                                                                                                                                        text language speaker_id chapter_id
/root/.cache/kagglehub/datasets/victorling/librispeech-clean/versions/1/LibriSpeech/train-clean-100/6272/70168/6272-70168-0000.flac                         INSTEAD OF MY HEART LEAPING FOR JOY AS I HAD EXPECTED A SUDDEN INDIFFERENCE FILLED IT I FELT AVERSE TO THE CHANGE FROM THE NARROW WAYS OF BARMOUTH WHICH FOR THE MOMENT I REGRETTED  English    

In [10]:
# ============================================================
# CELL 27E - CREATE ENGLISH TRAIN / VALIDATION SPLIT
# ============================================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_FILE = "/content/eldercare_kaggle_data/processed/english_asr_1000.csv"
OUTPUT_DIR = "/content/eldercare_kaggle_data/processed"

TRAIN_FILE = os.path.join(OUTPUT_DIR, "english_train.csv")
VAL_FILE = os.path.join(OUTPUT_DIR, "english_validation.csv")

print("========== CREATING TRAIN / VALIDATION SPLIT ==========")

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("Total samples:", len(df))
print("Total speakers:", df["speaker_id"].nunique())

# ------------------------------------------------------------
# Split by speaker
# ------------------------------------------------------------

speakers = df["speaker_id"].unique()

train_speakers, val_speakers = train_test_split(
    speakers,
    test_size=0.10,
    random_state=42
)

train_df = df[
    df["speaker_id"].isin(train_speakers)
].reset_index(drop=True)

val_df = df[
    df["speaker_id"].isin(val_speakers)
].reset_index(drop=True)

# ------------------------------------------------------------
# Save datasets
# ------------------------------------------------------------

train_df.to_csv(TRAIN_FILE, index=False)
val_df.to_csv(VAL_FILE, index=False)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n========== SPLIT SUMMARY ==========")

print("Training samples   :", len(train_df))
print("Validation samples :", len(val_df))

print("\nTraining speakers   :", train_df["speaker_id"].nunique())
print("Validation speakers :", val_df["speaker_id"].nunique())

# Check speaker overlap
overlap = set(train_df["speaker_id"]) & set(val_df["speaker_id"])

print("\nSpeaker overlap:", len(overlap))

if len(overlap) == 0:
    print("✓ No speaker overlap")
else:
    print("⚠️ Speaker overlap detected:", overlap)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

print("\n========== FILES CREATED ==========")

print("Training:")
print(TRAIN_FILE)
print("Exists:", os.path.exists(TRAIN_FILE))

print("\nValidation:")
print(VAL_FILE)
print("Exists:", os.path.exists(VAL_FILE))

print("\n================================================")
print("✓ TRAIN / VALIDATION SPLIT COMPLETED")
print("================================================")

========== CREATING TRAIN / VALIDATION SPLIT ==========
Total samples: 1000
Total speakers: 9

========== SPLIT SUMMARY ==========
Training samples   : 862
Validation samples : 138

Training speakers   : 8
Validation speakers : 1

Speaker overlap: 0
✓ No speaker overlap

========== FILES CREATED ==========
Training:
/content/eldercare_kaggle_data/processed/english_train.csv
Exists: True

Validation:
/content/eldercare_kaggle_data/processed/english_validation.csv
Exists: True

✓ TRAIN / VALIDATION SPLIT COMPLETED


In [11]:
# ============================================================
# CELL 27F - LOAD ENGLISH DATASET FOR MMS ASR
# ============================================================

import pandas as pd
from datasets import Dataset, Audio

TRAIN_FILE = "/content/eldercare_kaggle_data/processed/english_train.csv"
VAL_FILE = "/content/eldercare_kaggle_data/processed/english_validation.csv"

print("========== LOADING ENGLISH ASR DATA ==========")

# ------------------------------------------------------------
# Load CSV files
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

print("Training samples   :", len(train_df))
print("Validation samples :", len(val_df))

# ------------------------------------------------------------
# Convert Pandas → Hugging Face Dataset
# ------------------------------------------------------------

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

# ------------------------------------------------------------
# Convert audio column
# LibriSpeech audio is already 16 kHz
# ------------------------------------------------------------

train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

val_dataset = val_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

# ------------------------------------------------------------
# Display dataset information
# ------------------------------------------------------------

print("\n========== DATASET INFORMATION ==========")

print("\nTraining dataset:")
print(train_dataset)

print("\nValidation dataset:")
print(val_dataset)

print("\n========== FIRST TRAINING SAMPLE ==========")

sample = train_dataset[0]

print("Audio:")
print(sample["audio"])

print("\nTranscript:")
print(sample["text"])

print("\nLanguage:")
print(sample["language"])

print("\nSample rate:")
print(sample["audio"]["sampling_rate"])

print("\n================================================")
print("✓ ENGLISH DATASET READY FOR MMS")
print("================================================")

========== LOADING ENGLISH ASR DATA ==========
Training samples   : 862
Validation samples : 138

========== DATASET INFORMATION ==========

Training dataset:
Dataset({
    features: ['audio', 'text', 'language', 'speaker_id', 'chapter_id'],
    num_rows: 862
})

Validation dataset:
Dataset({
    features: ['audio', 'text', 'language', 'speaker_id', 'chapter_id'],
    num_rows: 138
})

========== FIRST TRAINING SAMPLE ==========
Audio:

Transcript:
INSTEAD OF MY HEART LEAPING FOR JOY AS I HAD EXPECTED A SUDDEN INDIFFERENCE FILLED IT I FELT AVERSE TO THE CHANGE FROM THE NARROW WAYS OF BARMOUTH WHICH FOR THE MOMENT I REGRETTED

Language:
English

Sample rate:
16000

✓ ENGLISH DATASET READY FOR MMS


In [13]:
# ============================================================
# CELL 27G - CORRECTED MMS ASR PREPROCESSING
# Compatible with Transformers 4.57.6
# ============================================================

import torch
from transformers import AutoProcessor

MODEL_NAME = "facebook/mms-1b-all"

print("========== LOADING MMS PROCESSOR ==========")

processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Select English tokenizer
processor.tokenizer.set_target_lang("eng")

print("Model:", MODEL_NAME)
print("Target language: English (eng)")
print("Processor loaded ✓")


# ------------------------------------------------------------
# Preprocessing function
# ------------------------------------------------------------

def prepare_dataset(batch):

    # -----------------------------
    # AUDIO
    # -----------------------------
    audio = batch["audio"]

    speech = audio["array"]

    # Convert waveform to MMS input values
    audio_inputs = processor(
        speech,
        sampling_rate=16000
    )

    batch["input_values"] = audio_inputs.input_values[0]

    # -----------------------------
    # TRANSCRIPT / LABELS
    # -----------------------------
    # Transformers 4.57.6 does not support
    # processor.as_target_processor().
    #
    # Directly use the tokenizer for CTC labels.

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch


# ------------------------------------------------------------
# PROCESS TRAINING DATA
# ------------------------------------------------------------

print("\n========== PROCESSING TRAINING DATA ==========")

train_processed = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    desc="Processing training audio"
)


# ------------------------------------------------------------
# PROCESS VALIDATION DATA
# ------------------------------------------------------------

print("\n========== PROCESSING VALIDATION DATA ==========")

val_processed = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Processing validation audio"
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n========== PREPROCESSING COMPLETE ==========")

print("\nTraining dataset:")
print(train_processed)

print("\nValidation dataset:")
print(val_processed)


# ------------------------------------------------------------
# CHECK FIRST SAMPLE
# ------------------------------------------------------------

print("\n========== FIRST PROCESSED SAMPLE ==========")

sample = train_processed[0]

print("Input values type :", type(sample["input_values"]))
print("Input values size :", len(sample["input_values"]))

print("Labels type       :", type(sample["labels"]))
print("Labels size       :", len(sample["labels"]))

print("\nFirst few labels:")
print(sample["labels"][:20])

print("\n================================================")
print("✓ MMS PREPROCESSING COMPLETED SUCCESSFULLY")
print("================================================")

========== LOADING MMS PROCESSOR ==========
Model: facebook/mms-1b-all
Target language: English (eng)
Processor loaded ✓

========== PROCESSING TRAINING DATA ==========


Processing training audio:   0%|          | 0/862 [00:00<?, ? examples/s]


========== PROCESSING VALIDATION DATA ==========


Processing validation audio:   0%|          | 0/138 [00:00<?, ? examples/s]


========== PREPROCESSING COMPLETE ==========

Training dataset:
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 862
})

Validation dataset:
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 138
})

========== FIRST PROCESSED SAMPLE ==========
Input values type : <class 'list'>
Input values size : 192880
Labels type       : <class 'list'>
Labels size       : 179

First few labels:
[3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 4]

✓ MMS PREPROCESSING COMPLETED SUCCESSFULLY


In [14]:
# ============================================================
# CELL 27H - LOAD MMS-1B MODEL + ENGLISH ADAPTER
# ============================================================

import torch
from transformers import Wav2Vec2ForCTC

MODEL_NAME = "facebook/mms-1b-all"

print("========== LOADING MMS-1B MODEL ==========")

# ------------------------------------------------------------
# Check GPU
# ------------------------------------------------------------

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("⚠️ GPU not available.")

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print("\nLoading MMS-1B...")

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True
)

# ------------------------------------------------------------
# Select English adapter
# ------------------------------------------------------------

print("\nLoading English language adapter...")

model.load_adapter("eng")

# ------------------------------------------------------------
# Move model to GPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("\n========== MODEL INFORMATION ==========")

print("Model:", MODEL_NAME)
print("Language adapter: eng")
print("Device:", device)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print("\n================================================")
print("✓ MMS-1B ENGLISH MODEL LOADED")
print("================================================")

========== LOADING MMS-1B MODEL ==========
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Loading MMS-1B...


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]


Loading English language adapter...


adapter.eng.safetensors: reconstructing file:   0%|          |  0.00B / 9.43MB            

adapter.eng.safetensors: downloading bytes:           |  0.00B            


========== MODEL INFORMATION ==========
Model: facebook/mms-1b-all
Language adapter: eng
Device: cuda
Total parameters: 964,845,850
Trainable parameters: 964,845,850

✓ MMS-1B ENGLISH MODEL LOADED


In [15]:
# ============================================================
# CELL 27I - FREEZE MMS-1B BACKBONE
# ============================================================

print("========== CONFIGURING MEMORY-EFFICIENT TRAINING ==========")

# Freeze the complete pretrained MMS model
for param in model.parameters():
    param.requires_grad = False

# Train only the CTC output head
for param in model.lm_head.parameters():
    param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%"
)

print("\nTrainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, tuple(param.shape))

print("\n================================================")
print("✓ BACKBONE FROZEN")
print("✓ ONLY CTC HEAD WILL BE TRAINED")
print("================================================")

========== CONFIGURING MEMORY-EFFICIENT TRAINING ==========
Total parameters: 964,845,850
Trainable parameters: 197,274
Trainable percentage: 0.0204%

Trainable layers:
lm_head.weight (154, 1280)
lm_head.bias (154,)

✓ BACKBONE FROZEN
✓ ONLY CTC HEAD WILL BE TRAINED


In [16]:
# ============================================================
# CELL 27J - DATA COLLATOR FOR CTC TRAINING
# ============================================================

from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        # Input audio
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Target transcripts
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Replace padding with -100 so CTC loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("========== DATA COLLATOR READY ==========")
print("Input: audio → input_values")
print("Target: transcript → labels")
print("Padding tokens: -100")
print("================================================")
print("✓ CTC DATA COLLATOR CREATED")
print("================================================")

========== DATA COLLATOR READY ==========
Input: audio → input_values
Target: transcript → labels
Padding tokens: -100
✓ CTC DATA COLLATOR CREATED


In [17]:
# ============================================================
# CELL 27K - VERIFY TRAINING BATCH
# ============================================================

print("========== VERIFYING TRAINING BATCH ==========")

# Take a small batch
sample_features = [
    train_processed[i]
    for i in range(min(4, len(train_processed)))
]

batch = data_collator(sample_features)

print("Batch keys:", list(batch.keys()))

print("\nInput values shape:")
print(batch["input_values"].shape)

print("\nLabels shape:")
print(batch["labels"].shape)

print("\nInput dtype:")
print(batch["input_values"].dtype)

print("\nLabels dtype:")
print(batch["labels"].dtype)

print("\nNumber of -100 padding labels:")
print((batch["labels"] == -100).sum().item())

print("\n================================================")
print("✓ TRAINING BATCH VERIFIED")
print("================================================")

========== VERIFYING TRAINING BATCH ==========
Batch keys: ['input_values', 'attention_mask', 'labels']

Input values shape:
torch.Size([4, 236160])

Labels shape:
torch.Size([4, 203])

Input dtype:
torch.float32

Labels dtype:
torch.int64

Number of -100 padding labels:
63

✓ TRAINING BATCH VERIFIED


In [19]:
# ============================================================
# CELL 27L - TRAINING CONFIGURATION (CORRECTED)
# ============================================================

from transformers import TrainingArguments

print("========== CONFIGURING TRAINING ==========")

training_args = TrainingArguments(
    output_dir="/content/eldercare_voice_ai/mms_english_model",

    # Training
    num_train_epochs=3,
    learning_rate=1e-4,

    # Memory-efficient training
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    # Logging
    logging_steps=25,

    # Performance
    fp16=True,
    gradient_checkpointing=False,

    # Best model
    load_best_model_at_end=False,

    # Reporting
    report_to="none",

    # Dataloader
    dataloader_num_workers=0,
)

print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)

print(
    "Effective batch size:",
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)

print("FP16:", training_args.fp16)
print("Evaluation:", training_args.eval_strategy)

print("\n================================================")
print("✓ TRAINING CONFIGURATION READY")
print("================================================")

========== CONFIGURING TRAINING ==========
Epochs: 3
Learning rate: 0.0001
Train batch size: 1
Gradient accumulation: 8
Effective batch size: 8
FP16: True
Evaluation: IntervalStrategy.EPOCH

✓ TRAINING CONFIGURATION READY


In [20]:
# ============================================================
# CELL 27M - CREATE MMS TRAINER
# ============================================================

from transformers import Trainer

print("========== CREATING TRAINER ==========")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    data_collator=data_collator,
)

print("\nTrainer model:", type(trainer.model).__name__)
print("Training samples:", len(train_processed))
print("Validation samples:", len(val_processed))

print("\n================================================")
print("✓ TRAINER CREATED SUCCESSFULLY")
print("================================================")

========== CREATING TRAINER ==========

Trainer model: Wav2Vec2ForCTC
Training samples: 862
Validation samples: 138

✓ TRAINER CREATED SUCCESSFULLY


In [21]:
# ============================================================
# CELL 27N - PRE-TRAINING FORWARD PASS CHECK
# ============================================================

print("========== PRE-TRAINING LOSS CHECK ==========")

model.train()

# Get one small batch
test_features = [
    train_processed[i]
    for i in range(2)
]

test_batch = data_collator(test_features)

# Move batch to GPU
test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

print("Input shape:", test_batch["input_values"].shape)
print("Labels shape:", test_batch["labels"].shape)

# Forward pass
with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
    outputs = model(**test_batch)

print("\nModel output shape:", outputs.logits.shape)

print("Initial CTC loss:", outputs.loss.item())

print("\n================================================")
print("✓ FORWARD PASS SUCCESSFUL")
print("✓ CTC LOSS CALCULATED")
print("================================================")

========== PRE-TRAINING LOSS CHECK ==========
Input shape: torch.Size([2, 212800])
Labels shape: torch.Size([2, 196])

Model output shape: torch.Size([2, 664, 154])
Initial CTC loss: 23.573944091796875

✓ FORWARD PASS SUCCESSFUL
✓ CTC LOSS CALCULATED


In [22]:
# ============================================================
# CELL 27O - TRAIN ENGLISH MMS ASR MODEL
# ============================================================

print("========== STARTING ENGLISH MMS TRAINING ==========")

print("Training samples:", len(train_processed))
print("Validation samples:", len(val_processed))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", 8)
print("Trainable parameters:", f"{trainable_params:,}")

print("\n⚠️ Training started...")
print("Please allow the cell to complete.\n")

train_result = trainer.train()

print("\n================================================")
print("✓ ENGLISH MMS TRAINING COMPLETED")
print("================================================")

print("\nTraining metrics:")
print(train_result.metrics)

========== STARTING ENGLISH MMS TRAINING ==========
Training samples: 862
Validation samples: 138
Epochs: 3
Effective batch size: 8
Trainable parameters: 197,274

⚠️ Training started...
Please allow the cell to complete.



Epoch,Training Loss,Validation Loss
1,119.103633,13.352555
2,72.498315,7.516359
3,50.470952,5.561906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ ENGLISH MMS TRAINING COMPLETED

Training metrics:
{'train_runtime': 3235.1899, 'train_samples_per_second': 0.799, 'train_steps_per_second': 0.1, 'total_flos': 3.1580799374473083e+18, 'train_loss': 92.67303504472898, 'epoch': 3.0}


In [24]:
# ============================================================
# CELL 27P - CORRECT ENGLISH ASR EVALUATION
# ============================================================

import torch
from jiwer import wer

print("========== CORRECT ENGLISH ASR EVALUATION ==========")

model.eval()

predictions = []
references = []

NUM_TEST_SAMPLES = min(20, len(val_processed))

for i in range(NUM_TEST_SAMPLES):

    sample = val_processed[i]

    # Original transcript from validation dataframe
    reference_text = str(val_df.iloc[i]["text"])

    input_values = torch.tensor(
        sample["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    attention_mask = torch.ones_like(
        input_values,
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits = model(
            input_values=input_values,
            attention_mask=attention_mask
        ).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    predicted_text = processor.batch_decode(
        predicted_ids
    )[0]

    predictions.append(predicted_text)
    references.append(reference_text)

    print(f"\nSample {i + 1}")
    print("Reference :", reference_text)
    print("Predicted :", predicted_text)

# Calculate actual WER
score = wer(references, predictions)

print("\n================================================")
print("CORRECT ENGLISH ASR EVALUATION")
print("================================================")
print("Samples tested:", NUM_TEST_SAMPLES)
print(f"WER: {score:.4f}")
print(f"WER (%): {score * 100:.2f}%")
print("================================================")

========== CORRECT ENGLISH ASR EVALUATION ==========

Sample 1
Reference : A FATHOM DEEP IN SLEEP I LIE WITH OLD DESIRES RESTRAINED BEFORE TO CLAMOR LIFEWARD WITH A CRY AS DARK FLIES OUT THE GREYING DOOR AND SO IN QUEST OF CREEDS TO SHARE I SEEK ASSERTIVE DAY AGAIN
Predicted : fathom deepnsleep ili with olddesire restraind before to clamor lifeward with a cry as dark flis ut te grayn door ansinquestof creds toshar i seek assertiv day gn

Sample 2
Reference : BUT OLD MONOTONY IS THERE ENDLESS AVENUES OF RAIN OH MIGHT I RISE AGAIN MIGHT I THROW OFF THE HEAT OF THAT OLD WINE SEE THE NEW MORNING MASS THE SKY WITH FAIRY TOWERS LINE ON LINE
Predicted : but oldmonotony is ter endlss avue of rain o mit i rise gin migt  throw off the heat of  old wine see the new morning ma te sky with fary tower lin nl

Sample 3
Reference : THE AIR BECAME GRAY AND OPALESCENT A SOLITARY LIGHT SUDDENLY OUTLINED A WINDOW OVER THE WAY THEN ANOTHER LIGHT THEN A HUNDRED MORE DANCED AND GLIMMERED INTO VISION UNDER HI

In [25]:
import re
from jiwer import wer

def normalize_text(text):
    text = text.upper()
    text = re.sub(r"[^A-Z0-9' ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("========== WER VERIFICATION ==========")

test_predictions = []
test_references = []

NUM_TEST_SAMPLES = 20

model.eval()

for i in range(NUM_TEST_SAMPLES):

    sample = val_processed[i]

    reference = str(val_df.iloc[i]["text"])

    input_values = torch.tensor(
        sample["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    attention_mask = torch.ones_like(
        input_values,
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits = model(
            input_values=input_values,
            attention_mask=attention_mask
        ).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(
        predicted_ids,
        group_tokens=True
    )[0]

    reference = normalize_text(reference)
    prediction = normalize_text(prediction)

    test_references.append(reference)
    test_predictions.append(prediction)

    print(f"\nSample {i+1}")
    print("REF :", reference)
    print("PRED:", prediction)

score = wer(
    test_references,
    test_predictions
)

print("\n======================================")
print("Samples tested:", NUM_TEST_SAMPLES)
print("WER:", score)
print("WER (%):", score * 100)
print("======================================")

========== WER VERIFICATION ==========

Sample 1
REF : A FATHOM DEEP IN SLEEP I LIE WITH OLD DESIRES RESTRAINED BEFORE TO CLAMOR LIFEWARD WITH A CRY AS DARK FLIES OUT THE GREYING DOOR AND SO IN QUEST OF CREEDS TO SHARE I SEEK ASSERTIVE DAY AGAIN
PRED: FATHOM DEEPNSLEEP ILI WITH OLDDESIRE RESTRAIND BEFORE TO CLAMOR LIFEWARD WITH A CRY AS DARK FLIS UT TE GRAYN DOOR ANSINQUESTOF CREDS TOSHAR I SEEK ASSERTIV DAY GN

Sample 2
REF : BUT OLD MONOTONY IS THERE ENDLESS AVENUES OF RAIN OH MIGHT I RISE AGAIN MIGHT I THROW OFF THE HEAT OF THAT OLD WINE SEE THE NEW MORNING MASS THE SKY WITH FAIRY TOWERS LINE ON LINE
PRED: BUT OLDMONOTONY IS TER ENDLSS AVUE OF RAIN O MIT I RISE GIN MIGT THROW OFF THE HEAT OF OLD WINE SEE THE NEW MORNING MA TE SKY WITH FARY TOWER LIN NL

Sample 3
REF : THE AIR BECAME GRAY AND OPALESCENT A SOLITARY LIGHT SUDDENLY OUTLINED A WINDOW OVER THE WAY THEN ANOTHER LIGHT THEN A HUNDRED MORE DANCED AND GLIMMERED INTO VISION UNDER HIS FEET A THICK IRON STUDDED SKYLIGHT TURNED YE

In [26]:
print("========== MMS ADAPTER PARAMETERS ==========")

adapter_params = []

for name, param in model.named_parameters():
    if "adapter" in name.lower():
        adapter_params.append((name, param))

print("Number of adapter parameter tensors:", len(adapter_params))

for name, param in adapter_params[:50]:
    print(f"{name:80s} {tuple(param.shape)}")

print("\nTotal adapter parameters:")

total_adapter_params = sum(
    param.numel() for _, param in adapter_params
)

print(total_adapter_params)

========== MMS ADAPTER PARAMETERS ==========
Number of adapter parameter tensors: 288
wav2vec2.encoder.layers.0.adapter_layer.norm.weight                              (1280,)
wav2vec2.encoder.layers.0.adapter_layer.norm.bias                                (1280,)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight                          (16, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias                            (16,)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight                          (1280, 16)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias                            (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.weight                              (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.bias                                (1280,)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.weight                          (16, 1280)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.bias                            (16,)
wav2vec2.encoder.la

In [27]:
# ==========================================
# ENABLE MMS ENGLISH ADAPTER + LM HEAD
# ==========================================

print("========== CONFIGURING TRAINABLE PARAMETERS ==========")

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Enable all MMS adapter parameters
adapter_trainable = 0
adapter_tensors = 0

for name, param in model.named_parameters():
    if "adapter_layer" in name:
        param.requires_grad = True
        adapter_trainable += param.numel()
        adapter_tensors += 1

# Enable CTC language-model head
lm_trainable = 0
lm_tensors = 0

for name, param in model.named_parameters():
    if "lm_head" in name:
        param.requires_grad = True
        lm_trainable += param.numel()
        lm_tensors += 1

total_params = sum(p.numel() for p in model.parameters())
total_trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters       : {total_params:,}")
print(f"Adapter parameters     : {adapter_trainable:,}")
print(f"LM head parameters     : {lm_trainable:,}")
print(f"Trainable parameters   : {total_trainable:,}")
print(f"Trainable percentage   : {100 * total_trainable / total_params:.4f}%")
print(f"Adapter tensors        : {adapter_tensors}")
print(f"LM head tensors        : {lm_tensors}")

print("\nTrainable parameter groups:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:80s} {tuple(param.shape)}")

========== CONFIGURING TRAINABLE PARAMETERS ==========
Total parameters       : 964,845,850
Adapter parameters     : 2,151,168
LM head parameters     : 197,274
Trainable parameters   : 2,348,442
Trainable percentage   : 0.2434%
Adapter tensors        : 288
LM head tensors        : 2

Trainable parameter groups:
wav2vec2.encoder.layers.0.adapter_layer.norm.weight                              (1280,)
wav2vec2.encoder.layers.0.adapter_layer.norm.bias                                (1280,)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight                          (16, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias                            (16,)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight                          (1280, 16)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias                            (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.weight                              (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.bias           

In [28]:
# ==========================================
# IMPROVED MMS ENGLISH ASR TRAINER
# Adapter + LM Head Fine-Tuning
# ==========================================

from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/eldercare_voice_ai/mms_english_adapter_model",

    num_train_epochs=3,

    learning_rate=1e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    logging_steps=25,

    fp16=True,

    gradient_checkpointing=False,

    load_best_model_at_end=False,

    report_to="none",

    dataloader_num_workers=0,
)

trainer_adapter = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    data_collator=data_collator,
)

print("========== IMPROVED TRAINER CREATED ==========")
print("Training samples :", len(train_processed))
print("Validation samples:", len(val_processed))
print("Trainable params :", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("==============================================")

========== IMPROVED TRAINER CREATED ==========
Training samples : 862
Validation samples: 138
Trainable params : 2348442


In [29]:
# ==========================================
# TRAIN MMS ENGLISH ADAPTER + LM HEAD
# ==========================================

print("========== STARTING ADAPTER TRAINING ==========")

train_result = trainer_adapter.train()

print("\n========== TRAINING COMPLETED ==========")
print("Training loss:", train_result.training_loss)
print("Training runtime:", train_result.metrics.get("train_runtime"))
print("========================================")


========== STARTING ADAPTER TRAINING ==========


Epoch,Training Loss,Validation Loss
1,5.167277,0.520592
2,3.755843,0.433481
3,3.449488,0.419541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


========== TRAINING COMPLETED ==========
Training loss: 8.19101604414575
Training runtime: 3585.3173


In [30]:
import torch
import re
from jiwer import wer

# ==========================================
# EVALUATE ADAPTER + LM HEAD MODEL
# ==========================================

model.eval()

predictions = []
references = []

print("========== STARTING ADAPTER EVALUATION ==========")

for i, sample in enumerate(val_processed.select(range(20))):

    input_values = torch.tensor(
        sample["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(
        predicted_ids,
        group_tokens=True
    )[0]

    # Get original reference from validation dataframe
    reference = val_df.iloc[i]["text"]

    predictions.append(prediction)
    references.append(reference)

    print(f"\nSample {i+1}")
    print("REF :", reference)
    print("PRED:", prediction)

# ==========================================
# NORMALIZE TEXT
# ==========================================

def normalize_text(text):
    text = text.upper()
    text = re.sub(r"[^A-Z0-9' ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

normalized_refs = [
    normalize_text(x) for x in references
]

normalized_preds = [
    normalize_text(x) for x in predictions
]

# ==========================================
# CALCULATE WER
# ==========================================

adapter_wer = wer(
    normalized_refs,
    normalized_preds
)

print("\n==============================================")
print("ADAPTER + LM HEAD MODEL RESULTS")
print("==============================================")
print(f"WER: {adapter_wer:.4f}")
print(f"WER percentage: {adapter_wer * 100:.2f}%")
print("==============================================")

========== STARTING ADAPTER EVALUATION ==========

Sample 1
REF : A FATHOM DEEP IN SLEEP I LIE WITH OLD DESIRES RESTRAINED BEFORE TO CLAMOR LIFEWARD WITH A CRY AS DARK FLIES OUT THE GREYING DOOR AND SO IN QUEST OF CREEDS TO SHARE I SEEK ASSERTIVE DAY AGAIN
PRED: <unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk> <unk><unk><unk><unk><unk><unk> <unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk> <unk><unk> <unk> <unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk> <unk><unk><unk><unk> <unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk> <unk><unk> <unk><unk><unk><unk><unk><unk><unk> <unk> <unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk> <un

In [31]:
# ==========================================
# CHECK MMS TOKENIZER / <unk> CONFIGURATION
# ==========================================

print("========== MMS TOKENIZER CHECK ==========")

print("Tokenizer class :", type(processor.tokenizer).__name__)
print("Vocab size      :", len(processor.tokenizer))

print("UNK token       :", processor.tokenizer.unk_token)
print("UNK token ID    :", processor.tokenizer.unk_token_id)

print("PAD token       :", processor.tokenizer.pad_token)
print("PAD token ID    :", processor.tokenizer.pad_token_id)

print("BOS token       :", processor.tokenizer.bos_token)
print("EOS token       :", processor.tokenizer.eos_token)

print("\nModel vocab size:", model.config.vocab_size)
print("LM head output  :", model.lm_head.out_features)

# Test a simple English sentence
test_text = "I AM AWAKE"

encoded = processor.tokenizer(test_text)

print("\nTest text       :", test_text)
print("Encoded IDs     :", encoded.input_ids)

decoded = processor.tokenizer.decode(
    encoded.input_ids,
    group_tokens=False
)

print("Decoded text    :", decoded)

# Check first few vocabulary entries
vocab = processor.tokenizer.get_vocab()

print("\nFirst 20 vocabulary entries:")
for token, token_id in list(vocab.items())[:20]:
    print(token_id, repr(token))

print("========================================")

========== MMS TOKENIZER CHECK ==========
Tokenizer class : Wav2Vec2CTCTokenizer
Vocab size      : 154
UNK token       : <unk>
UNK token ID    : 3
PAD token       : <pad>
PAD token ID    : 0
BOS token       : <s>
EOS token       : </s>

Model vocab size: 154
LM head output  : 154

Test text       : I AM AWAKE
Encoded IDs     : [3, 4, 3, 3, 4, 3, 3, 3, 3, 3]
Decoded text    : <unk> <unk><unk> <unk><unk><unk><unk><unk>

First 20 vocabulary entries:
75 '!'
62 '"'
48 '$'
47 '%'
57 '&'
31 "'"
56 '+'
44 ','
32 '-'
37 '.'
45 '/'
27 '0'
30 '1'
35 '2'
41 '3'
39 '4'
38 '5'
40 '6'
43 '7'
42 '8'


In [32]:

# ==========================================
# CHECK MMS ENGLISH TOKENIZER
# ==========================================

test_sentences = [
    "I AM AWAKE",
    "i am awake",
    "THE AIR BECAME GRAY",
    "the air became gray"
]

print("========== TOKENIZER CASE TEST ==========")

for text in test_sentences:
    ids = processor.tokenizer(text).input_ids
    decoded = processor.tokenizer.decode(
        ids,
        group_tokens=False
    )

    print("\nTEXT   :", text)
    print("IDS    :", ids)
    print("DECODE :", decoded)

print("\n========== CHARACTER VOCAB CHECK ==========")

for char in ["a", "b", "c", "i", "m", "t", "h", "e", " "]:
    token_id = processor.tokenizer.convert_tokens_to_ids(char)
    print(repr(char), "->", token_id)

print("==========================================")

========== TOKENIZER CASE TEST ==========

TEXT   : I AM AWAKE
IDS    : [3, 4, 3, 3, 4, 3, 3, 3, 3, 3]
DECODE : <unk> <unk><unk> <unk><unk><unk><unk><unk>

TEXT   : i am awake
IDS    : [9, 4, 7, 18, 4, 7, 23, 7, 26, 5]
DECODE : i am awake

TEXT   : THE AIR BECAME GRAY
IDS    : [3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3]
DECODE : <unk><unk><unk> <unk><unk><unk> <unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk>

TEXT   : the air became gray
IDS    : [6, 13, 5, 4, 7, 9, 12, 4, 24, 5, 16, 7, 18, 5, 4, 21, 12, 7, 22]
DECODE : the air became gray

========== CHARACTER VOCAB CHECK ==========
'a' -> 7
'b' -> 24
'c' -> 16
'i' -> 9
'm' -> 18
't' -> 6
'h' -> 13
'e' -> 5
' ' -> 3


In [33]:

# ==========================================
# REBUILD ASR DATASET WITH LOWERCASE TEXT
# ==========================================

from datasets import Dataset, Audio

print("========== REBUILDING LOWERCASE DATASET ==========")

# Reload original CSV files
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

# Convert transcripts to lowercase
train_df["text"] = train_df["text"].astype(str).str.lower()
val_df["text"] = val_df["text"].astype(str).str.lower()

# Create HF datasets
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

# Make sure audio is decoded at 16 kHz
train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

val_dataset = val_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))

print("\nExample transcript:")
print("Original CSV text :", pd.read_csv(TRAIN_FILE).iloc[0]["text"])
print("New training text :", train_dataset[0]["text"])

print("\n========== DATASET READY ==========")

========== REBUILDING LOWERCASE DATASET ==========
Training samples   : 862
Validation samples : 138

Example transcript:
Original CSV text : INSTEAD OF MY HEART LEAPING FOR JOY AS I HAD EXPECTED A SUDDEN INDIFFERENCE FILLED IT I FELT AVERSE TO THE CHANGE FROM THE NARROW WAYS OF BARMOUTH WHICH FOR THE MOMENT I REGRETTED
New training text : instead of my heart leaping for joy as i had expected a sudden indifference filled it i felt averse to the change from the narrow ways of barmouth which for the moment i regretted

========== DATASET READY ==========


In [34]:

# ==========================================
# PREPROCESS LOWERCASE ASR DATASET
# ==========================================

print("========== PREPROCESSING LOWERCASE DATA ==========")

def prepare_dataset(batch):
    audio = batch["audio"]

    # Audio → model input
    speech = audio["array"]

    audio_inputs = processor(
        speech,
        sampling_rate=16000
    )

    batch["input_values"] = audio_inputs.input_values[0]

    # Lowercase transcript → tokenizer labels
    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch


train_processed = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    desc="Processing training audio"
)

val_processed = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    desc="Processing validation audio"
)

print("\n========== PREPROCESSING COMPLETE ==========")
print("Training samples   :", len(train_processed))
print("Validation samples :", len(val_processed))

print("\nFeatures:")
print(train_processed.features)

print("\nFirst transcript:")
print(train_dataset[0]["text"])

print("\nFirst label IDs:")
print(train_processed[0]["labels"])

print("============================================")

========== PREPROCESSING LOWERCASE DATA ==========


Processing training audio:   0%|          | 0/862 [00:00<?, ? examples/s]

Processing validation audio:   0%|          | 0/138 [00:00<?, ? examples/s]


========== PREPROCESSING COMPLETE ==========
Training samples   : 862
Validation samples : 138

Features:
{'input_values': List(Value('float32')), 'labels': List(Value('int64'))}

First transcript:
instead of my heart leaping for joy as i had expected a sudden indifference filled it i felt averse to the change from the narrow ways of barmouth which for the moment i regretted

First label IDs:
[9, 10, 11, 6, 5, 7, 15, 4, 8, 19, 4, 18, 22, 4, 13, 5, 7, 12, 6, 4, 14, 5, 7, 20, 9, 10, 21, 4, 19, 8, 12, 4, 29, 8, 22, 4, 7, 11, 4, 9, 4, 13, 7, 15, 4, 5, 28, 20, 5, 16, 6, 5, 15, 4, 7, 4, 11, 17, 15, 15, 5, 10, 4, 9, 10, 15, 9, 19, 19, 5, 12, 5, 10, 16, 5, 4, 19, 9, 14, 14, 5, 15, 4, 9, 6, 4, 9, 4, 19, 5, 14, 6, 4, 7, 25, 5, 12, 11, 5, 4, 6, 8, 4, 6, 13, 5, 4, 16, 13, 7, 10, 21, 5, 4, 19, 12, 8, 18, 4, 6, 13, 5, 4, 10, 7, 12, 12, 8, 23, 4, 23, 7, 22, 11, 4, 8, 19, 4, 24, 7, 12, 18, 8, 17, 6, 13, 4, 23, 13, 9, 16, 13, 4, 19, 8, 12, 4, 6, 13, 5, 4, 18, 8, 18, 5, 10, 6, 4, 9, 4, 12, 5, 21, 12, 5

In [35]:
print("========== LABEL SANITY CHECK ==========")

# Check tokenizer decoding
sample_text = train_dataset[0]["text"]
sample_labels = train_processed[0]["labels"]

decoded_text = processor.tokenizer.decode(sample_labels)

print("Original text:")
print(sample_text)

print("\nDecoded labels:")
print(decoded_text)

print("\nExact match:")
print(sample_text == decoded_text)

# Count UNK tokens
unk_id = processor.tokenizer.unk_token_id

total_labels = sum(len(x["labels"]) for x in train_processed)
unk_labels = sum(
    sum(label == unk_id for label in x["labels"])
    for x in train_processed
)

print("\nTotal label tokens :", total_labels)
print("UNK label tokens   :", unk_labels)
print("UNK percentage     :", round((unk_labels / total_labels) * 100, 4), "%")

print("========================================")

========== LABEL SANITY CHECK ==========
Original text:
instead of my heart leaping for joy as i had expected a sudden indifference filled it i felt averse to the change from the narrow ways of barmouth which for the moment i regretted

Decoded labels:
instead of my heart leaping for joy as i had expected a suden indiference filed it i felt averse to the change from the narow ways of barmouth which for the moment i regreted

Exact match:
False

Total label tokens : 170168
UNK label tokens   : 0
UNK percentage     : 0.0 %


In [36]:
print("========== CTC BATCH SANITY CHECK ==========")

# Create the data collator
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

# Test a small batch
test_batch = data_collator([
    train_processed[0],
    train_processed[1],
    train_processed[2],
    train_processed[3]
])

print("Batch keys:")
print(test_batch.keys())

print("\nInput values shape:")
print(test_batch["input_values"].shape)

print("\nAttention mask shape:")
print(test_batch["attention_mask"].shape)

print("\nLabels shape:")
print(test_batch["labels"].shape)

print("\nInput dtype:")
print(test_batch["input_values"].dtype)

print("\nLabels dtype:")
print(test_batch["labels"].dtype)

print("\nNumber of -100 labels:")
print((test_batch["labels"] == -100).sum().item())

print("\nNumber of UNK labels:")
print((test_batch["labels"] == processor.tokenizer.unk_token_id).sum().item())

print("==============================================")

========== CTC BATCH SANITY CHECK ==========
Batch keys:
KeysView({'input_values': tensor([[ 0.0205,  0.0160, -0.0019,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0004,  0.0018,  0.0056,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0371,  0.0146, -0.0051,  ..., -0.0065, -0.0446, -0.0440],
        [-0.0514, -0.0442, -0.0265,  ...,  0.0000,  0.0000,  0.0000]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), 'labels': tensor([[   9,   10,   11,    6,    5,    7,   15,    4,    8,   19,    4,   18,
           22,    4,   13,    5,    7,   12,    6,    4,   14,    5,    7,   20,
            9,   10,   21,    4,   19,    8,   12,    4,   29,    8,   22,    4,
            7,   11,    4,    9,    4,   13,    7,   15,    4,    5,   28,   20,
            5,   16,    6,    5,   15,    4,    7,    4,   11,   17,   15,   15,
            5,   10,    4,    9,   10,   15, 

In [37]:
print("========== LOADING FRESH MMS ENGLISH MODEL ==========")

import torch
from transformers import Wav2Vec2ForCTC

# Load fresh pretrained MMS model
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b-all",
    target_lang="eng",
    ignore_mismatched_sizes=True
)

# Load English adapter
model.load_adapter("eng")

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("\nModel loaded successfully")
print("Device :", device)
print("Total parameters :", sum(p.numel() for p in model.parameters()))
print("Trainable parameters :", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

print("======================================================")

========== LOADING FRESH MMS ENGLISH MODEL ==========


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]


Model loaded successfully
Device : cuda
Total parameters : 964845850
Trainable parameters : 964845850


In [38]:
print("========== MMS FORWARD PASS TEST ==========")

# Put model in evaluation mode for this test
model.eval()

# Take 2 corrected training samples
test_batch = data_collator([
    train_processed[0],
    train_processed[1]
])

# Move tensors to GPU
input_values = test_batch["input_values"].to(device)
attention_mask = test_batch["attention_mask"].to(device)
labels = test_batch["labels"].to(device)

print("Input shape :", input_values.shape)
print("Labels shape:", labels.shape)

# Forward pass
with torch.no_grad():
    outputs = model(
        input_values=input_values,
        attention_mask=attention_mask,
        labels=labels
    )

print("\nLogits shape:")
print(outputs.logits.shape)

print("\nCTC loss:")
print(outputs.loss.item())

print("\nLoss valid:")
print(torch.isfinite(outputs.loss).item())

print("==========================================")

========== MMS FORWARD PASS TEST ==========
Input shape : torch.Size([2, 212800])
Labels shape: torch.Size([2, 196])

Logits shape:
torch.Size([2, 664, 154])

CTC loss:
0.0464070700109005

Loss valid:
True


In [39]:
print("========== CONFIGURING MMS FINE-TUNING ==========")

# Freeze the entire model
for param in model.parameters():
    param.requires_grad = False

# Enable MMS language adapter
for name, param in model.named_parameters():
    if "adapter_layer" in name:
        param.requires_grad = True

# Enable CTC language-model head
for name, param in model.named_parameters():
    if "lm_head" in name:
        param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
adapter_params = sum(
    p.numel() for name, p in model.named_parameters()
    if "adapter_layer" in name
)
lm_head_params = sum(
    p.numel() for name, p in model.named_parameters()
    if "lm_head" in name
)
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters       :", total_params)
print("Adapter parameters     :", adapter_params)
print("LM head parameters     :", lm_head_params)
print("Trainable parameters   :", trainable_params)
print(
    "Trainable percentage    :",
    round(trainable_params / total_params * 100, 4),
    "%"
)

print("\nTrainable parameter groups:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, tuple(param.shape))

print("=================================================")

========== CONFIGURING MMS FINE-TUNING ==========
Total parameters       : 964845850
Adapter parameters     : 2151168
LM head parameters     : 197274
Trainable parameters   : 2348442
Trainable percentage    : 0.2434 %

Trainable parameter groups:
wav2vec2.encoder.layers.0.adapter_layer.norm.weight (1280,)
wav2vec2.encoder.layers.0.adapter_layer.norm.bias (1280,)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight (16, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias (16,)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight (1280, 16)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.weight (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.bias (1280,)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.weight (16, 1280)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.bias (16,)
wav2vec2.encoder.layers.1.adapter_layer.linear_2.weight (1280, 16)
wav2vec2.encoder.layers.1.adapter_layer.linear_2.bias (1280,)
wav2v

In [40]:
print("========== CREATING CORRECTED TRAINER ==========")

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/eldercare_voice_ai/mms_english_corrected",
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=25,
    fp16=True,
    gradient_checkpointing=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    data_collator=data_collator,
)

print("\n========== TRAINER CREATED ==========")
print("Training samples      :", len(train_processed))
print("Validation samples    :", len(val_processed))
print("Trainable params      :", trainable_params)
print("Epochs                :", training_args.num_train_epochs)
print("Learning rate         :", training_args.learning_rate)
print("Batch size            :", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("=====================================")

========== CREATING CORRECTED TRAINER ==========

========== TRAINER CREATED ==========
Training samples      : 862
Validation samples    : 138
Trainable params      : 2348442
Epochs                : 3
Learning rate         : 0.0001
Batch size            : 1
Gradient accumulation: 8


In [41]:
print("Training dataset:")
print(train_dataset)

print("\nValidation dataset:")
print(val_dataset)


Training dataset:
Dataset({
    features: ['audio', 'text', 'language', 'speaker_id', 'chapter_id'],
    num_rows: 862
})

Validation dataset:
Dataset({
    features: ['audio', 'text', 'language', 'speaker_id', 'chapter_id'],
    num_rows: 138
})


In [42]:
print("Training languages:")
print(train_dataset.unique("language"))

print("\nValidation languages:")
print(val_dataset.unique("language"))

Training languages:
['English']

Validation languages:
['English']
